# RappiPlus | From Data to Business Decisions

## Project Overview

This project evaluates the performance of **RappiPlus** to support **data-driven business decisions**. It combines transactional, product, marketing, behavioral, and experimental data to assess commercial performance and identify opportunities for improvement.

### Data Sources

- **`rappiplus_orders_raw.csv`** — Orders, prices, discounts, and revenue
- **`rappiplus_catalog.csv`** — Product costs, categories, and suppliers
- **`rappiplus_marketing_spend.csv`** — Marketing expenditure by channel and country
- **`events`, `users`, `user_activity` (SQL)** — User behavior on the platform
- **`experiment_checkout_ui.csv`** — Checkout A/B test results

### Analytical Workflow

1. **Data quality:** Assess the reliability of the source data using Python.
2. **Business profitability:** Analyze revenue, product costs, and estimated profit.
3. **Conversion funnel:** Identify where users drop off during their journey.
4. **Cohort retention:** Examine whether users return to the platform.
5. **Statistical testing:** Evaluate the evidence for an effect from a checkout experiment.
6. **Business intelligence:** Communicate results through interactive Power BI dashboards.

The goal is to translate validated data into interpretable findings and **actionable business recommendations**.


---

## Step 1 | Data Loading and Quality Validation

### 1.1 Data Loading and Initial Exploration

**Objective:** Understand the structure of the three commercial datasets before conducting the analysis.

The required Python libraries are imported, and the source files (`rappiplus_orders_raw.csv`, `rappiplus_catalog.csv`, and `rappiplus_marketing_spend.csv`) are loaded into the `orders`, `catalog`, and `marketing` DataFrames. Initial previews and dataset dimensions are then inspected.

---


In [ ]:
# Import libraries
import pandas as pd

In [ ]:
# Load source files
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# Explore datasets

print('orders')
display(orders.head())
print('Dimensiones:', orders.shape)
orders.info()


In [ ]:
# Explore the product catalog
print('catalog')
display(catalog.head())
print('Dimensiones:', catalog.shape)
catalog.info()

catalog


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


Dimensiones: (7, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [ ]:
# Explore marketing data
print('marketing')
display(marketing.head())
print('Dimensiones:', marketing.shape)
marketing.info()


marketing


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


Dimensiones: (1620, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Data Quality Assessment

**Objective:** Identify and address data issues that could affect revenue, cost, and profitability calculations.

The assessment covers all three datasets and includes:

- Validating date fields and converting them to appropriate data types
- Checking numeric fields for potentially invalid negative or zero values
- Verifying consistency across transaction amounts
- Identifying and removing exact duplicates
- Inspecting categorical fields and missing values

---


In [ ]:
# 1. Validación de variables numéricas y conversión de fechas
# Diagnóstico de fecha_hora_pedido

print("Tipo de dato:")
print(orders['fecha_hora_pedido'].dtype)

print("\nPrimeros valores:")
print(orders['fecha_hora_pedido'].head(10))

print("\nEjemplos de tipos almacenados:")
print(orders['fecha_hora_pedido'].map(type).value_counts())

print("\nValores nulos:")
print(orders['fecha_hora_pedido'].isna().sum())

Tipo de dato:
object

Primeros valores:
0    2025-05-22
1    2025-06-15
2    2025-05-02
3    2025-06-09
4    2025-03-30
5    2025-04-22
6    2025-03-15
7    2025-05-01
8    2025-03-01
9    2025-05-31
Name: fecha_hora_pedido, dtype: object

Ejemplos de tipos almacenados:
<class 'str'>    25100
Name: fecha_hora_pedido, dtype: int64

Valores nulos:
0


In [ ]:
# Convert dates using an explicit format

orders['fecha_hora_pedido'] = pd.to_datetime(
    orders['fecha_hora_pedido'],
    format='%Y-%m-%d',
    errors='coerce'
)

marketing['fecha'] = pd.to_datetime(
    marketing['fecha'],
    format='%Y-%m-%d',
    errors='coerce'
)

# Validar conversión
print("Tipo de fecha en orders:", orders['fecha_hora_pedido'].dtype)
print("Fechas inválidas en orders:", orders['fecha_hora_pedido'].isna().sum())

print("\nTipo de fecha en marketing:", marketing['fecha'].dtype)
print("Fechas inválidas en marketing:", marketing['fecha'].isna().sum())

Tipo de fecha en orders: datetime64[ns]
Fechas inválidas en orders: 0

Tipo de fecha en marketing: datetime64[ns]
Fechas inválidas en marketing: 0


In [ ]:
# 2. Revisión de variables numéricas

print("\nVariables numéricas - orders")
display(
    orders[
        ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
    ].describe()
)

print("Variables numéricas - catalog")
display(catalog[['costo_unitario']].describe())

print("Variables numéricas - marketing")
display(marketing[['gasto']].describe())



Variables numéricas - orders


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


Variables numéricas - catalog


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


Variables numéricas - marketing


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [ ]:
# Check potentially invalid negative or zero values

print("Orders")
print("Cantidad <= 0:", (orders['cantidad'] <= 0).sum())
print("Precio unitario <= 0:", (orders['precio_unitario'] <= 0).sum())
print("Monto descuento < 0:", (orders['monto_descuento'] < 0).sum())
print("Monto total <= 0:", (orders['monto_total'] <= 0).sum())

print("\nCatalog")
print("Costo unitario <= 0:", (catalog['costo_unitario'] <= 0).sum())

print("\nMarketing")
print("Gasto < 0:", (marketing['gasto'] < 0).sum())

Orders
Cantidad <= 0: 4
Precio unitario <= 0: 0
Monto descuento < 0: 0
Monto total <= 0: 4

Catalog
Costo unitario <= 0: 0

Marketing
Gasto < 0: 0


In [ ]:
# Inspect records with invalid quantities or amounts

invalid_orders = orders[
    (orders['cantidad'] <= 0) |
    (orders['monto_total'] <= 0)
]

print("Registros con valores inválidos:")
display(invalid_orders)


# Revisar cantidades más altas

print("\nPedidos con las cantidades más altas:")
display(
    orders.nlargest(
        10, 'cantidad'
    )[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

Registros con valores inválidos:


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53



Pedidos con las cantidades más altas:


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
3656,order_3656,Laptop-Gaming-16GB,20000.0,297.66,0.0,5953200.0
3668,order_3668,Laptop-Gaming-16GB,20000.0,348.31,0.0,6966200.0
3722,order_3722,Laptop-Gaming-16GB,20000.0,442.01,0.0,8840200.0
3726,order_3726,Laptop-Gaming-16GB,20000.0,290.85,0.0,5817000.0
3521,order_3521,Laptop-Gaming-16GB,10000.0,43.14,0.0,431400.0
3522,order_3522,Laptop-Gaming-16GB,10000.0,280.55,0.0,2805500.0
3586,order_3586,Laptop-Gaming-16GB,10000.0,490.35,0.0,4903500.0
3643,order_3643,Laptop-Gaming-16GB,10000.0,238.15,0.0,2381500.0
3689,order_3689,Laptop-Gaming-16GB,10000.0,87.69,0.0,876900.0
3748,order_3748,Laptop-Gaming-16GB,10000.0,336.93,0.0,3369300.0


In [ ]:
# Inspect quantity distribution

print("Frecuencia de cantidades:")
print(
    orders['cantidad']
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPedidos con cantidad > 10:")
display(
    orders[orders['cantidad'] > 10][
        [
            'id_pedido',
            'nombre_producto',
            'categoria_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

Frecuencia de cantidades:
-2.0            1
-1.0            3
 1.0        12394
 2.0        12642
 10000.0        6
 20000.0        4
 NaN           50
Name: cantidad, dtype: int64

Pedidos con cantidad > 10:


,id_pedido,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.0
3522,order_3522,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.0
3586,order_3586,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.0
3643,order_3643,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.0
3656,order_3656,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.0
3668,order_3668,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.0
3689,order_3689,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.0
3722,order_3722,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.0
3726,order_3726,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.0
3748,order_3748,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.0


In [ ]:
# Check missing values in numeric fields

numeric_cols = [
    'cantidad',
    'precio_unitario',
    'monto_descuento'
]

print("Nulos por columna:")
print(orders[numeric_cols].isna().sum())

missing_numeric = orders[
    orders[numeric_cols].isna().any(axis=1)
]

print("\nFilas con al menos un valor numérico faltante:")
print(len(missing_numeric))

display(missing_numeric.head(10))

Nulos por columna:
cantidad           50
precio_unitario    50
monto_descuento    50
dtype: int64

Filas con al menos un valor numérico faltante:
50


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,desktop,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,desktop,organic,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,mobile,paid_search,Tablet-Standard-64GB,NaN,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,mobile,paid_search,Sneakers-Urban-42,NaN,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,279.35


In [ ]:
# Verificar si los productos de las filas con faltantes
# existen en el catálogo

missing_products = missing_numeric[
    ['nombre_producto']
].drop_duplicates()

display(
    missing_products.merge(
        catalog[
            ['nombre_producto', 'categoria_producto', 'costo_unitario']
        ],
        on='nombre_producto',
        how='left'
    )
)

,nombre_producto,categoria_producto,costo_unitario
0,Sneakers-Urban-42,Moda,17.21
1,Laptop-Gaming-16GB,Electrónica,280.68
2,Vacuum-Pro-Black,Hogar,16.60
3,Phone-Pro-128GB,Electrónica,10.12
4,Tablet-Standard-64GB,Electrónica,25.21
5,Jacket-Winter-M,Moda,189.31
6,Blender-XL-Red,Hogar,176.64


In [ ]:
# Revisar valores habituales por producto

product_summary = (
    orders
    .dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])
    .groupby('nombre_producto')
    .agg(
        pedidos=('id_pedido', 'count'),
        cantidad_mediana=('cantidad', 'median'),
        precio_mediano=('precio_unitario', 'median'),
        descuento_mediano=('monto_descuento', 'median')
    )
    .reset_index()
)

display(product_summary)

,nombre_producto,pedidos,cantidad_mediana,precio_mediano,descuento_mediano
0,Blender-XL-Red,4192,2.0,256.335,5.0
1,Jacket-Winter-M,4182,2.0,259.090,5.0
2,Laptop-Gaming-16GB,2790,2.0,260.745,5.0
3,Phone-Pro-128GB,2747,2.0,261.320,0.0
4,Sneakers-Urban-42,4141,1.0,256.020,0.0
5,Tablet-Standard-64GB,2775,2.0,255.320,0.0
6,Vacuum-Pro-Black,4193,2.0,261.490,0.0


In [ ]:
# Inspect typical behavior for each product
product_summary = (
    orders
    .dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])
    .groupby('nombre_producto')
    .agg(
        pedidos=('id_pedido', 'count'),
        cantidad_mediana=('cantidad', 'median'),
        precio_mediano=('precio_unitario', 'median'),
        descuento_mediano=('monto_descuento', 'median')
    )
    .reset_index()
)

display(product_summary)

,nombre_producto,pedidos,cantidad_mediana,precio_mediano,descuento_mediano
0,Blender-XL-Red,4192,2.0,256.335,5.0
1,Jacket-Winter-M,4182,2.0,259.090,5.0
2,Laptop-Gaming-16GB,2790,2.0,260.745,5.0
3,Phone-Pro-128GB,2747,2.0,261.320,0.0
4,Sneakers-Urban-42,4141,1.0,256.020,0.0
5,Tablet-Standard-64GB,2775,2.0,255.320,0.0
6,Vacuum-Pro-Black,4193,2.0,261.490,0.0


In [ ]:
# Validate consistency of monto_total

orders['monto_calculado'] = (
    orders['cantidad'] * orders['precio_unitario']
    - orders['monto_descuento']
)

orders['diferencia_monto'] = (
    orders['monto_total'] - orders['monto_calculado']
).abs()

# Resumen de las diferencias
print(orders['diferencia_monto'].describe())

# Cantidad de registros con diferencias relevantes
print(
    "\nRegistros con diferencia > 0.02:",
    (orders['diferencia_monto'] > 0.02).sum()
)

# Revisar los casos con mayor diferencia
display(
    orders[
        ['id_pedido', 'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total',
         'monto_calculado', 'diferencia_monto']
    ]
    .sort_values('diferencia_monto', ascending=False)
    .head(10)
)

count    2.505000e+04
mean     4.094212e-03
std      1.547943e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      2.000000e+01
Name: diferencia_monto, dtype: float64

Registros con diferencia > 0.02: 3


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado,diferencia_monto
266,order_266,-2.0,101.31,10.0,-192.62,-212.62,20.00
268,order_268,-1.0,497.65,5.0,-492.65,-502.65,10.00
267,order_267,-1.0,43.50,5.0,-38.50,-48.50,10.00
18851,order_18851,2.0,325.84,10.0,641.69,641.68,0.01
1784,order_1784,2.0,447.34,10.0,884.69,884.68,0.01
12359,order_12359,2.0,368.97,0.0,737.93,737.94,0.01
19469,order_19469,2.0,487.66,0.0,975.31,975.32,0.01
14051,order_14051,2.0,477.34,15.0,939.69,939.68,0.01
5936,order_5936,2.0,324.66,15.0,634.31,634.32,0.01
1022,order_1022,2.0,269.66,0.0,539.31,539.32,0.01


In [ ]:
display(
    orders.loc[
        orders['cantidad'] < 0,
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'nombre_producto', 'cantidad',
         'precio_unitario', 'monto_descuento', 'monto_total']
    ]
)

,id_pedido,id_usuario,fecha_hora_pedido,pais,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,Phone-Pro-128GB,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,Phone-Pro-128GB,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,Phone-Pro-128GB,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,Phone-Pro-128GB,-1.0,423.53,0.0,-423.53


In [ ]:
extremos = orders[orders['cantidad'] > 10].copy()

extremos['monto_esperado'] = (
    extremos['cantidad'] * extremos['precio_unitario']
    - extremos['monto_descuento']
)

extremos['diferencia'] = (
    extremos['monto_total'] - extremos['monto_esperado']
).abs()

display(
    extremos[
        ['id_pedido', 'nombre_producto', 'cantidad',
         'precio_unitario', 'monto_descuento',
         'monto_total', 'monto_esperado', 'diferencia']
    ]
)

,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_esperado,diferencia
3521,order_3521,Laptop-Gaming-16GB,10000.0,43.14,0.0,431400.0,431400.0,0.000000e+00
3522,order_3522,Laptop-Gaming-16GB,10000.0,280.55,0.0,2805500.0,2805500.0,0.000000e+00
3586,order_3586,Laptop-Gaming-16GB,10000.0,490.35,0.0,4903500.0,4903500.0,0.000000e+00
3643,order_3643,Laptop-Gaming-16GB,10000.0,238.15,0.0,2381500.0,2381500.0,0.000000e+00
3656,order_3656,Laptop-Gaming-16GB,20000.0,297.66,0.0,5953200.0,5953200.0,9.313226e-10
3668,order_3668,Laptop-Gaming-16GB,20000.0,348.31,0.0,6966200.0,6966200.0,0.000000e+00
3689,order_3689,Laptop-Gaming-16GB,10000.0,87.69,0.0,876900.0,876900.0,0.000000e+00
3722,order_3722,Laptop-Gaming-16GB,20000.0,442.01,0.0,8840200.0,8840200.0,0.000000e+00
3726,order_3726,Laptop-Gaming-16GB,20000.0,290.85,0.0,5817000.0,5817000.0,0.000000e+00
3748,order_3748,Laptop-Gaming-16GB,10000.0,336.93,0.0,3369300.0,3369300.0,0.000000e+00


In [ ]:
# Columnas originales relevantes
cols_originales = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto',
    'cantidad',
    'precio_unitario',
    'monto_descuento'
]

# Cantidad de nulos
print("Nulos por columna:")
print(orders[cols_originales].isna().sum())

# Patrones de nulos
patrones_nulos = (
    orders[cols_originales]
    .isna()
    .value_counts()
    .reset_index(name='cantidad_filas')
)

display(patrones_nulos)

Nulos por columna:
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
dtype: int64


,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,cantidad_filas
0,False,False,False,False,False,False,False,False,24700
1,True,False,False,False,False,False,False,False,300
2,False,False,False,False,True,True,True,True,50
3,False,False,True,True,True,False,False,False,30
4,False,True,False,False,False,False,False,False,20


In [ ]:
# Check for duplicates

print("Duplicados exactos:")
print("Orders:", orders.duplicated().sum())
print("Catalog:", catalog.duplicated().sum())
print("Marketing:", marketing.duplicated().sum())

print("\nIDs de pedido duplicados:")
print(orders['id_pedido'].duplicated().sum())

print("\nProductos duplicados en catálogo:")
print(catalog['nombre_producto'].duplicated().sum())

Duplicados exactos:
Orders: 100
Catalog: 0
Marketing: 0

IDs de pedido duplicados:
100

Productos duplicados en catálogo:
0


In [ ]:
# Inspect categorical variables

categoricas_orders = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]

for col in categoricas_orders:
    print(f"\n--- {col} ---")
    print(orders[col].value_counts(dropna=False))


--- pais ---
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

--- dispositivo ---
desktop    12759
mobile     12321
NaN           20
Name: dispositivo, dtype: int64

--- fuente_referencia ---
social         8428
organic        8329
paid_search    8313
NaN              30
Name: fuente_referencia, dtype: int64

--- nombre_producto ---
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
NaN                       30
Name: nombre_producto, dtype: int64

--- categoria_producto ---
Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: categoria_producto, dtype: int64


In [ ]:
for col in ['pais', 'id_campaña', 'canal']:
    print(f"\n--- {col} ---")
    print(marketing[col].value_counts(dropna=False))


--- pais ---
Colombia     540
Argentina    540
Mexico       540
Name: pais, dtype: int64

--- id_campaña ---
paid_search_Colombia     180
organic_Colombia         180
social_Mexico            180
organic_Mexico           180
social_Colombia          180
social_Argentina         180
organic_Argentina        180
paid_search_Mexico       180
paid_search_Argentina    180
Name: id_campaña, dtype: int64

--- canal ---
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64


In [ ]:
# Mostrar algunos pedidos duplicados
duplicados_orders = orders[
    orders.duplicated(subset='id_pedido', keep=False)
].sort_values('id_pedido')

display(duplicados_orders.head(20))

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_calculado,diferencia_monto
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67,442.68,0.01
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67,442.68,0.01
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10,160.10,0.00
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10,160.10,0.00
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08,231.08,0.00
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08,231.08,0.00
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53,72.53,0.00
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53,72.53,0.00
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55,55.55,0.00
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55,55.55,0.00


In [ ]:
marketing.loc[
    marketing['canal'].isna(),
    ['fecha', 'pais', 'id_campaña', 'canal', 'gasto']
].head(20)

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
103,2025-01-12,Colombia,paid_search_Colombia,NaN,1583.33
104,2025-01-12,Colombia,social_Colombia,NaN,712.64
105,2025-01-12,Argentina,organic_Argentina,NaN,1860.40
106,2025-01-12,Argentina,paid_search_Argentina,NaN,885.06
107,2025-01-12,Argentina,social_Argentina,NaN,2253.77


In [ ]:
# Remove exact duplicates
orders = orders.drop_duplicates().copy()

# Validar resultado
print("Filas después de eliminar duplicados:", len(orders))
print("Duplicados exactos restantes:", orders.duplicated().sum())
print("IDs de pedido duplicados restantes:",
      orders['id_pedido'].duplicated().sum())

Filas después de eliminar duplicados: 25000
Duplicados exactos restantes: 0
IDs de pedido duplicados restantes: 0


In [ ]:
# Comprobar correspondencia entre id_campaña y canal
canales_campaña = (
    marketing
    .dropna(subset=['canal'])
    .groupby('id_campaña')['canal']
    .unique()
)

display(canales_campaña)

id_campaña
organic_Argentina            [organic]
organic_Colombia             [organic]
organic_Mexico               [organic]
paid_search_Argentina    [paid_search]
paid_search_Colombia     [paid_search]
paid_search_Mexico       [paid_search]
social_Argentina              [social]
social_Colombia               [social]
social_Mexico                 [social]
Name: canal, dtype: object

In [ ]:
# Crear mapa campaña -> canal con los registros completos
mapa_canal = (
    marketing
    .dropna(subset=['canal'])
    .drop_duplicates('id_campaña')
    .set_index('id_campaña')['canal']
)

# Completar únicamente los canales faltantes
marketing['canal'] = marketing['canal'].fillna(
    marketing['id_campaña'].map(mapa_canal)
)

# Validar
print("Canales faltantes:", marketing['canal'].isna().sum())
print("\nDistribución final:")
print(marketing['canal'].value_counts())

Canales faltantes: 0

Distribución final:
organic        540
paid_search    540
social         540
Name: canal, dtype: int64


In [ ]:
# Standardize country names
orders['pais'] = orders['pais'].str.strip().str.title()

print("Distribución de países:")
print(orders['pais'].value_counts(dropna=False))

Distribución de países:
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64


In [ ]:
# Inspect records with missing country information
pais_faltante = orders[orders['pais'].isna()]

print("Cantidad de países faltantes:", len(pais_faltante))

display(
    pais_faltante[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'dispositivo', 'fuente_referencia',
         'nombre_producto', 'categoria_producto',
         'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total']
    ].head(20)
)

Cantidad de países faltantes: 300


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
124,order_124,user_6671,2025-04-12,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,298.44,5.0,293.44
125,order_125,user_5263,2025-05-30,NaN,mobile,organic,Tablet-Standard-64GB,Electronica,1.0,133.13,0.0,133.13
126,order_126,user_4952,2025-04-19,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,2.0,177.23,0.0,354.45
127,order_127,user_2619,2025-06-26,NaN,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,329.53,10.0,319.53
128,order_128,user_2444,2025-05-06,NaN,desktop,social,Vacuum-Pro-Black,Hogar,2.0,218.43,0.0,436.86
129,order_129,user_7157,2025-04-17,NaN,desktop,paid_search,Sneakers-Urban-42,Moda,1.0,367.88,10.0,357.88
130,order_130,user_3279,2025-02-06,NaN,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,114.83,0.0,114.83
131,order_131,user_2914,2025-05-05,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,117.84,10.0,107.84
132,order_132,user_2173,2025-02-03,NaN,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,383.84,0.0,383.84
133,order_133,user_4038,2025-05-12,NaN,mobile,social,Vacuum-Pro-Black,Hogar,2.0,345.85,0.0,691.69


In [ ]:
# Países conocidos por usuario
usuarios_pais = (
    orders
    .dropna(subset=['pais'])
    .groupby('id_usuario')['pais']
    .nunique()
)

print("Usuarios asociados a un solo país:")
print((usuarios_pais == 1).sum())

print("\nUsuarios asociados a más de un país:")
print((usuarios_pais > 1).sum())

Usuarios asociados a un solo país:
7624

Usuarios asociados a más de un país:
0


In [ ]:
# Crear mapa usuario -> país a partir de registros con país conocido
mapa_usuario_pais = (
    orders
    .dropna(subset=['pais'])
    .drop_duplicates(subset=['id_usuario'])
    .set_index('id_usuario')['pais']
)

# Revisar cuántos países faltantes pueden recuperarse por id_usuario
pais_faltante = orders[orders['pais'].isna()].copy()

pais_faltante['pais_recuperado'] = (
    pais_faltante['id_usuario'].map(mapa_usuario_pais)
)

print("Total de países faltantes:", len(pais_faltante))
print(
    "Recuperables por id_usuario:",
    pais_faltante['pais_recuperado'].notna().sum()
)
print(
    "No recuperables por id_usuario:",
    pais_faltante['pais_recuperado'].isna().sum()
)

display(
    pais_faltante[
        ['id_pedido', 'id_usuario', 'pais', 'pais_recuperado']
    ].head(20)
)

Total de países faltantes: 300
Recuperables por id_usuario: 281
No recuperables por id_usuario: 19


,id_pedido,id_usuario,pais,pais_recuperado
124,order_124,user_6671,NaN,Mexico
125,order_125,user_5263,NaN,Argentina
126,order_126,user_4952,NaN,Mexico
127,order_127,user_2619,NaN,Mexico
128,order_128,user_2444,NaN,Mexico
129,order_129,user_7157,NaN,Mexico
130,order_130,user_3279,NaN,Mexico
131,order_131,user_2914,NaN,NaN
132,order_132,user_2173,NaN,Mexico
133,order_133,user_4038,NaN,Argentina


In [ ]:
# Recuperar país únicamente cuando existe información
# del mismo usuario en otros pedidos
orders['pais'] = orders['pais'].fillna(
    orders['id_usuario'].map(mapa_usuario_pais)
)

# Validar
print("Países faltantes después de recuperar por usuario:",
      orders['pais'].isna().sum())

print("\nDistribución actual:")
print(orders['pais'].value_counts(dropna=False))

Países faltantes después de recuperar por usuario: 19

Distribución actual:
Mexico       8440
Colombia     8386
Argentina    8155
NaN            19
Name: pais, dtype: int64


In [ ]:
# Examinar los países que no pudieron recuperarse
pais_sin_recuperar = orders[orders['pais'].isna()]

print("Registros sin país:", len(pais_sin_recuperar))

display(
    pais_sin_recuperar[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'dispositivo', 'fuente_referencia',
         'nombre_producto', 'categoria_producto',
         'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total']
    ]
)

Registros sin país: 19


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
131,order_131,user_2914,2025-05-05,NaN,mobile,organic,Blender-XL-Red,Hogar,1.0,117.84,10.0,107.84
172,order_172,user_1918,2025-01-14,NaN,desktop,organic,Laptop-Gaming-16GB,Electronica,1.0,490.44,10.0,480.44
173,order_173,user_3776,2025-05-14,NaN,desktop,paid_search,Jacket-Winter-M,Moda,2.0,319.58,0.0,639.16
188,order_188,user_7144,2025-01-26,NaN,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,384.46,15.0,369.46
189,order_189,user_722,2025-03-15,NaN,mobile,paid_search,Vacuum-Pro-Black,Hogar,1.0,405.64,5.0,400.64
212,order_212,user_7893,2025-03-19,NaN,mobile,paid_search,Vacuum-Pro-Black,Hogar,2.0,376.60,10.0,743.20
224,order_224,user_4192,2025-04-08,NaN,mobile,social,Jacket-Winter-M,Moda,1.0,119.20,0.0,119.20
228,order_228,user_4917,2025-05-25,NaN,mobile,social,Laptop-Gaming-16GB,Electronica,1.0,302.95,10.0,292.95
275,order_275,user_3523,2025-03-12,NaN,desktop,social,Sneakers-Urban-42,Moda,1.0,467.46,15.0,452.46
281,order_281,user_7788,2025-06-14,NaN,desktop,paid_search,Blender-XL-Red,Hogar,2.0,100.68,0.0,201.35


In [ ]:
# Los países recuperables ya fueron completados mediante id_usuario.
# Los 19 casos sin evidencia suficiente se conservan como Desconocido.

orders['pais'] = orders['pais'].fillna('Desconocido')

# Validación
print("Valores nulos en país:", orders['pais'].isna().sum())

print("\nDistribución final de países:")
print(orders['pais'].value_counts())

print(
    "\nRegistros clasificados como Desconocido:",
    (orders['pais'] == 'Desconocido').sum()
)

Valores nulos en país: 0

Distribución final de países:
Mexico         8440
Colombia       8386
Argentina      8155
Desconocido      19
Name: pais, dtype: int64

Registros clasificados como Desconocido: 19


In [ ]:
# Inspect records with missing device information
dispositivo_faltante = orders[orders['dispositivo'].isna()]

print("Dispositivos faltantes:", len(dispositivo_faltante))

display(
    dispositivo_faltante[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'dispositivo', 'fuente_referencia',
         'nombre_producto', 'categoria_producto',
         'cantidad', 'monto_total']
    ]
)

Dispositivos faltantes: 20


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,monto_total
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,235.99
25,order_25,user_540,2025-03-08,Argentina,NaN,social,Vacuum-Pro-Black,Hogar,1.0,178.50
26,order_26,user_1951,2025-02-10,Colombia,NaN,organic,Blender-XL-Red,Hogar,1.0,316.52
27,order_27,user_5410,2025-05-09,Colombia,NaN,paid_search,Phone-Pro-128GB,Electronica,2.0,893.96
28,order_28,user_1489,2025-01-17,Mexico,NaN,social,Phone-Pro-128GB,Electronica,1.0,50.70
29,order_29,user_5197,2025-01-30,Colombia,NaN,social,Phone-Pro-128GB,Electronica,1.0,381.57
30,order_30,user_5386,2025-01-31,Mexico,NaN,social,Vacuum-Pro-Black,Hogar,1.0,173.74
31,order_31,user_5766,2025-05-05,Colombia,NaN,paid_search,Vacuum-Pro-Black,Hogar,1.0,480.74
32,order_32,user_4400,2025-03-29,Mexico,NaN,social,Tablet-Standard-64GB,Electronica,1.0,117.69
33,order_33,user_5070,2025-01-12,Argentina,NaN,paid_search,Tablet-Standard-64GB,Electronica,1.0,86.82


In [ ]:
# Número de dispositivos conocidos por usuario
usuarios_dispositivo = (
    orders
    .dropna(subset=['dispositivo'])
    .groupby('id_usuario')['dispositivo']
    .nunique()
)

print("Usuarios asociados a un solo dispositivo:")
print((usuarios_dispositivo == 1).sum())

print("\nUsuarios asociados a más de un dispositivo:")
print((usuarios_dispositivo > 1).sum())

Usuarios asociados a un solo dispositivo:
7640

Usuarios asociados a más de un dispositivo:
0


In [ ]:
# Historial conocido de los usuarios cuyo dispositivo está ausente
usuarios_con_dispositivo_faltante = dispositivo_faltante['id_usuario'].unique()

historial_dispositivo = (
    orders[
        orders['id_usuario'].isin(usuarios_con_dispositivo_faltante)
        & orders['dispositivo'].notna()
    ]
    .groupby('id_usuario')['dispositivo']
    .agg(
        dispositivos_conocidos=lambda x: sorted(x.unique()),
        cantidad_dispositivos='nunique',
        pedidos_conocidos='count'
    )
)

display(historial_dispositivo)

,dispositivos_conocidos,cantidad_dispositivos,pedidos_conocidos
id_usuario,,,
user_1489,[desktop],1,1
user_1951,[mobile],1,3
user_28,[mobile],1,1
user_2963,[desktop],1,6
user_3817,[desktop],1,4
user_3944,[mobile],1,1
user_4024,[mobile],1,2
user_4400,[desktop],1,4
user_5070,[mobile],1,8


In [ ]:
# Crear mapa usuario -> dispositivo
mapa_usuario_dispositivo = (
    orders
    .dropna(subset=['dispositivo'])
    .drop_duplicates(subset=['id_usuario'])
    .set_index('id_usuario')['dispositivo']
)

# Recuperar dispositivo mediante el historial del usuario
orders['dispositivo'] = orders['dispositivo'].fillna(
    orders['id_usuario'].map(mapa_usuario_dispositivo)
)

# Validar
print("Dispositivos faltantes después de recuperar por usuario:",
      orders['dispositivo'].isna().sum())

print("\nDistribución actual:")
print(orders['dispositivo'].value_counts(dropna=False))

Dispositivos faltantes después de recuperar por usuario: 2

Distribución actual:
desktop    12721
mobile     12277
NaN            2
Name: dispositivo, dtype: int64


In [ ]:
# Revisar los dispositivos que no pudieron recuperarse
dispositivo_sin_recuperar = orders[orders['dispositivo'].isna()]

display(
    dispositivo_sin_recuperar[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'dispositivo', 'fuente_referencia',
         'nombre_producto', 'categoria_producto',
         'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total']
    ]
)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99
34,order_34,user_579,2025-05-23,Mexico,NaN,social,Sneakers-Urban-42,Moda,1.0,174.85,0.0,174.85


In [ ]:
# No existe evidencia suficiente para inferir
# los dos dispositivos restantes
orders['dispositivo'] = orders['dispositivo'].fillna('Desconocido')

# Validación final
print("Dispositivos faltantes:", orders['dispositivo'].isna().sum())

print("\nDistribución final:")
print(orders['dispositivo'].value_counts())

Dispositivos faltantes: 0

Distribución final:
desktop        12721
mobile         12277
Desconocido        2
Name: dispositivo, dtype: int64


**Handling missing values in `dispositivo`:**

Of the 20 records with missing device information, 18 were recovered using the purchase history of the same user. For these users, the available records consistently indicated a single device type.

The remaining two records lacked sufficient history for reliable imputation. An additional comparison between `fuente_referencia` and `dispositivo` showed an approximately even split between `desktop` and `mobile` across channels. Since this did not provide a defensible basis for inference, the two values were classified as `Desconocido`.


In [ ]:
# Revisar registros con fuente de referencia faltante
fuente_faltante = orders[orders['fuente_referencia'].isna()]

print("Fuentes de referencia faltantes:", len(fuente_faltante))

display(
    fuente_faltante[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'dispositivo', 'fuente_referencia',
         'nombre_producto', 'categoria_producto',
         'cantidad', 'monto_total']
    ]
)

Fuentes de referencia faltantes: 30


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,472.98


In [ ]:
# Número de fuentes conocidas por usuario
usuarios_fuente = (
    orders
    .dropna(subset=['fuente_referencia'])
    .groupby('id_usuario')['fuente_referencia']
    .nunique()
)

print("Usuarios asociados a una sola fuente:")
print((usuarios_fuente == 1).sum())

print("\nUsuarios asociados a más de una fuente:")
print((usuarios_fuente > 1).sum())

Usuarios asociados a una sola fuente:
1932

Usuarios asociados a más de una fuente:
5710


In [ ]:
# Historial de fuentes de los usuarios con fuente faltante
usuarios_fuente_faltante = fuente_faltante['id_usuario'].unique()

historial_fuentes_faltantes = (
    orders[
        orders['id_usuario'].isin(usuarios_fuente_faltante)
        & orders['fuente_referencia'].notna()
    ]
    .groupby('id_usuario')
    .agg(
        fuentes_conocidas=('fuente_referencia', lambda x: list(x.unique())),
        cantidad_fuentes=('fuente_referencia', 'nunique'),
        pedidos_conocidos=('fuente_referencia', 'count')
    )
)

display(historial_fuentes_faltantes)

print("Usuarios con fuente faltante:", len(usuarios_fuente_faltante))

print(
    "Sin historial de fuente:",
    len(usuarios_fuente_faltante) - len(historial_fuentes_faltantes)
)

print(
    "Con una sola fuente histórica:",
    (historial_fuentes_faltantes['cantidad_fuentes'] == 1).sum()
)

print(
    "Con múltiples fuentes históricas:",
    (historial_fuentes_faltantes['cantidad_fuentes'] > 1).sum()
)

,fuentes_conocidas,cantidad_fuentes,pedidos_conocidos
id_usuario,,,
user_1741,"[social, organic]",2,4
user_1840,"[paid_search, social]",2,2
user_2319,"[organic, social]",2,3
user_2495,"[paid_search, organic, social]",3,5
user_2685,"[paid_search, social]",2,3
user_2899,[social],1,2
user_2923,[paid_search],1,1
user_3431,[organic],1,1
user_3507,"[paid_search, social]",2,5


Usuarios con fuente faltante: 30
Sin historial de fuente: 0
Con una sola fuente histórica: 10
Con múltiples fuentes históricas: 20


In [ ]:
# Usuarios de los registros que todavía tienen fuente faltante
usuarios_faltantes = orders.loc[
    orders['fuente_referencia'].isna(),
    'id_usuario'
].unique()

# Revisar específicamente su historial conocido
historial_30 = (
    orders[
        orders['id_usuario'].isin(usuarios_faltantes)
        & orders['fuente_referencia'].notna()
    ]
    .groupby('id_usuario')['fuente_referencia']
    .agg(
        fuentes=lambda x: list(x.unique()),
        cantidad_fuentes='nunique'
    )
)

display(historial_30)

print(
    "\nUsuarios con exactamente una fuente:",
    (historial_30['cantidad_fuentes'] == 1).sum()
)

,fuentes,cantidad_fuentes
id_usuario,,
user_1741,"[social, organic]",2
user_1840,"[paid_search, social]",2
user_2319,"[organic, social]",2
user_2495,"[paid_search, organic, social]",3
user_2685,"[paid_search, social]",2
user_2899,[social],1
user_2923,[paid_search],1
user_3431,[organic],1
user_3507,"[paid_search, social]",2



Usuarios con exactamente una fuente: 10


In [ ]:
# Crear mapa solamente para los usuarios con una fuente histórica
mapa_fuente = (
    historial_30[
        historial_30['cantidad_fuentes'] == 1
    ]['fuentes']
    .apply(lambda x: x[0])
)

display(mapa_fuente)

id_usuario
user_2899         social
user_2923    paid_search
user_3431        organic
user_4030    paid_search
user_5186    paid_search
user_5634        organic
user_5815         social
user_6018    paid_search
user_6592        organic
user_834          social
Name: fuentes, dtype: object

In [ ]:
# Ver qué fuente se recuperaría antes de modificar el DataFrame
comprobacion_fuente = orders.loc[
    orders['fuente_referencia'].isna(),
    ['id_pedido', 'id_usuario', 'fuente_referencia']
].copy()

comprobacion_fuente['fuente_recuperada'] = (
    comprobacion_fuente['id_usuario'].map(mapa_fuente)
)

display(comprobacion_fuente)

print(
    "Fuentes recuperables:",
    comprobacion_fuente['fuente_recuperada'].notna().sum()
)

,id_pedido,id_usuario,fuente_referencia,fuente_recuperada
44,order_44,user_2899,NaN,social
45,order_45,user_6196,NaN,NaN
46,order_46,user_5815,NaN,social
47,order_47,user_406,NaN,NaN
48,order_48,user_6187,NaN,NaN
49,order_49,user_7225,NaN,NaN
50,order_50,user_1840,NaN,NaN
51,order_51,user_5998,NaN,NaN
52,order_52,user_5679,NaN,NaN
53,order_53,user_5936,NaN,NaN


Fuentes recuperables: 10


In [ ]:
# Recuperar las fuentes que tienen evidencia suficiente
mask_fuente = orders['fuente_referencia'].isna()

orders.loc[mask_fuente, 'fuente_referencia'] = (
    orders.loc[mask_fuente, 'id_usuario']
    .map(mapa_fuente)
)

# Validar resultado
print(
    "Fuentes faltantes después de recuperar por usuario:",
    orders['fuente_referencia'].isna().sum()
)

print("\nDistribución actual:")
print(orders['fuente_referencia'].value_counts(dropna=False))

Fuentes faltantes después de recuperar por usuario: 20

Distribución actual:
social         8405
organic        8291
paid_search    8284
NaN              20
Name: fuente_referencia, dtype: int64


In [ ]:
# Los casos ambiguos se conservan como desconocidos
orders['fuente_referencia'] = (
    orders['fuente_referencia'].fillna('Desconocido')
)

# Validación final
print("Fuentes faltantes:", orders['fuente_referencia'].isna().sum())
print("\nDistribución final:")
print(orders['fuente_referencia'].value_counts())

Fuentes faltantes: 0

Distribución final:
social         8405
organic        8291
paid_search    8284
Desconocido      20
Name: fuente_referencia, dtype: int64


**Handling missing values in `fuente_referencia`:**

Thirty records had a missing referral source. User purchase histories showed that ten of these records belonged to users with only one known referral source, allowing those values to be recovered.

The remaining twenty records belonged to users associated with multiple referral sources. Without sufficient evidence to identify the source for each transaction, they were classified as `Desconocido` rather than assigning an arbitrary value.


In [ ]:
# Revisar patrones de valores faltantes entre producto y categoría
producto_categoria_nulos = pd.crosstab(
    orders['nombre_producto'].isna(),
    orders['categoria_producto'].isna()
)

display(producto_categoria_nulos)

categoria_producto,False,True
nombre_producto,,
False,24920,50
True,0,30


In [ ]:
# Casos donde falta nombre del producto
producto_faltante = orders[orders['nombre_producto'].isna()]

print("Productos faltantes:", len(producto_faltante))

display(
    producto_faltante[
        ['id_pedido', 'id_usuario', 'fecha_hora_pedido',
         'pais', 'nombre_producto', 'categoria_producto',
         'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total']
    ]
)

Productos faltantes: 30


,id_pedido,id_usuario,fecha_hora_pedido,pais,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,NaN,NaN,2.0,137.96,0.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,NaN,NaN,2.0,433.76,0.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,NaN,NaN,1.0,495.38,5.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,NaN,NaN,2.0,382.94,0.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,NaN,NaN,2.0,415.01,0.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,NaN,NaN,2.0,236.49,0.0,472.98


In [ ]:
# Crear mapa producto -> categoría desde catalog
mapa_categoria = (
    catalog
    .set_index('nombre_producto')['categoria_producto']
)

# Registros con producto conocido pero categoría faltante
mask_categoria = (
    orders['categoria_producto'].isna()
    & orders['nombre_producto'].notna()
)

comprobacion_categoria = orders.loc[
    mask_categoria,
    ['id_pedido', 'nombre_producto', 'categoria_producto']
].copy()

# Buscar categoría correspondiente en catálogo
comprobacion_categoria['categoria_recuperada'] = (
    comprobacion_categoria['nombre_producto']
    .map(mapa_categoria)
)

display(comprobacion_categoria)

print(
    "Categorías recuperables:",
    comprobacion_categoria['categoria_recuperada'].notna().sum()
)

print(
    "Categorías no recuperables:",
    comprobacion_categoria['categoria_recuperada'].isna().sum()
)

,id_pedido,nombre_producto,categoria_producto,categoria_recuperada
74,order_74,Sneakers-Urban-42,NaN,Moda
75,order_75,Sneakers-Urban-42,NaN,Moda
76,order_76,Laptop-Gaming-16GB,NaN,Electrónica
77,order_77,Vacuum-Pro-Black,NaN,Hogar
78,order_78,Phone-Pro-128GB,NaN,Electrónica
79,order_79,Sneakers-Urban-42,NaN,Moda
80,order_80,Vacuum-Pro-Black,NaN,Hogar
81,order_81,Tablet-Standard-64GB,NaN,Electrónica
82,order_82,Sneakers-Urban-42,NaN,Moda
83,order_83,Sneakers-Urban-42,NaN,Moda


Categorías recuperables: 50
Categorías no recuperables: 0


In [ ]:
# Recuperar categoría a partir del producto conocido
orders.loc[mask_categoria, 'categoria_producto'] = (
    orders.loc[mask_categoria, 'nombre_producto']
    .map(mapa_categoria)
)

# Validar
print(
    "Categorías faltantes después de recuperar por producto:",
    orders['categoria_producto'].isna().sum()
)

Categorías faltantes después de recuperar por producto: 30


In [ ]:
# Usuarios de los pedidos con producto faltante
usuarios_producto_faltante = orders.loc[
    orders['nombre_producto'].isna(),
    'id_usuario'
].unique()

historial_productos = (
    orders[
        orders['id_usuario'].isin(usuarios_producto_faltante)
        & orders['nombre_producto'].notna()
    ]
    .groupby('id_usuario')['nombre_producto']
    .agg(
        productos_conocidos=lambda x: list(x.unique()),
        cantidad_productos='nunique',
        pedidos_conocidos='count'
    )
)

display(historial_productos)

print(
    "\nUsuarios con un solo producto histórico:",
    (historial_productos['cantidad_productos'] == 1).sum()
)

print(
    "Usuarios con múltiples productos históricos:",
    (historial_productos['cantidad_productos'] > 1).sum()
)

print(
    "Usuarios sin historial de producto:",
    len(usuarios_producto_faltante) - len(historial_productos)
)

,productos_conocidos,cantidad_productos,pedidos_conocidos
id_usuario,,,
user_1741,"[Sneakers-Urban-42, Vacuum-Pro-Black, Laptop-G...",4,4
user_1840,"[Phone-Pro-128GB, Jacket-Winter-M]",2,2
user_2319,"[Laptop-Gaming-16GB, Blender-XL-Red, Phone-Pro...",3,3
user_2495,"[Jacket-Winter-M, Laptop-Gaming-16GB, Sneakers...",4,5
user_2685,"[Tablet-Standard-64GB, Blender-XL-Red]",2,3
user_2899,"[Tablet-Standard-64GB, Phone-Pro-128GB]",2,2
user_2923,[Phone-Pro-128GB],1,1
user_3431,[Tablet-Standard-64GB],1,1
user_3507,"[Sneakers-Urban-42, Laptop-Gaming-16GB, Phone-...",3,5



Usuarios con un solo producto histórico: 8
Usuarios con múltiples productos históricos: 22
Usuarios sin historial de producto: 0


In [ ]:
# Resumen de precios observados por producto
precios_producto = (
    orders
    .dropna(subset=['nombre_producto', 'precio_unitario'])
    .groupby('nombre_producto')['precio_unitario']
    .agg(
        pedidos='count',
        precio_min='min',
        precio_mediano='median',
        precio_max='max',
        precio_promedio='mean'
    )
    .round(2)
    .sort_values('precio_mediano')
)

display(precios_producto)

,pedidos,precio_min,precio_mediano,precio_max,precio_promedio
nombre_producto,,,,,
Tablet-Standard-64GB,2764,20.03,255.01,499.89,257.53
Sneakers-Urban-42,4129,20.11,256.18,499.91,257.23
Blender-XL-Red,4176,20.06,256.34,499.94,259.16
Jacket-Winter-M,4166,20.06,259.20,499.93,259.82
Laptop-Gaming-16GB,2778,20.24,261.16,499.95,261.36
Phone-Pro-128GB,2737,20.08,261.57,499.86,259.54
Vacuum-Pro-Black,4170,20.37,261.68,499.96,261.02


In [ ]:
# Revisar información económica de los pedidos sin producto
productos_sin_identificar = orders[
    orders['nombre_producto'].isna()
][
    [
        'id_pedido',
        'id_usuario',
        'pais',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
]

display(productos_sin_identificar)

print(
    "\nRango de precios de productos desconocidos:",
    productos_sin_identificar['precio_unitario'].min(),
    "-",
    productos_sin_identificar['precio_unitario'].max()
)

,id_pedido,id_usuario,pais,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,Colombia,1.0,318.38,5.0,313.38
45,order_45,user_6196,Mexico,2.0,354.06,10.0,698.12
46,order_46,user_5815,Mexico,1.0,359.31,10.0,349.31
47,order_47,user_406,Colombia,2.0,476.78,0.0,953.56
48,order_48,user_6187,Colombia,2.0,137.96,0.0,275.93
49,order_49,user_7225,Argentina,2.0,433.76,0.0,867.53
50,order_50,user_1840,Mexico,1.0,495.38,5.0,490.38
51,order_51,user_5998,Mexico,2.0,382.94,0.0,765.89
52,order_52,user_5679,Argentina,2.0,415.01,0.0,830.01
53,order_53,user_5936,Argentina,2.0,236.49,0.0,472.98



Rango de precios de productos desconocidos: 26.86 - 495.38


In [ ]:
# Analizar precios por producto y país
precios_producto_pais = (
    orders
    .dropna(subset=['nombre_producto', 'pais', 'precio_unitario'])
    .groupby(['pais', 'nombre_producto'])['precio_unitario']
    .agg(
        pedidos='count',
        precio_min='min',
        precio_mediano='median',
        precio_max='max',
        precio_promedio='mean'
    )
    .round(2)
    .reset_index()
)

display(precios_producto_pais)

,pais,nombre_producto,pedidos,precio_min,precio_mediano,precio_max,precio_promedio
0,Argentina,Blender-XL-Red,1329,20.36,260.84,499.27,262.36
1,Argentina,Jacket-Winter-M,1353,20.60,270.18,499.86,262.17
2,Argentina,Laptop-Gaming-16GB,916,20.30,277.71,499.95,267.97
3,Argentina,Phone-Pro-128GB,868,20.40,262.62,499.67,261.64
4,Argentina,Sneakers-Urban-42,1386,20.11,251.23,499.89,255.20
5,Argentina,Tablet-Standard-64GB,916,20.03,240.79,497.99,250.14
6,Argentina,Vacuum-Pro-Black,1357,20.87,267.33,499.96,261.31
7,Colombia,Blender-XL-Red,1400,20.06,259.63,499.93,260.06
8,Colombia,Jacket-Winter-M,1447,20.06,257.00,499.93,256.43
9,Colombia,Laptop-Gaming-16GB,930,20.27,245.82,499.78,252.74


In [ ]:
# Pedidos sin producto
sin_producto = orders[
    orders['nombre_producto'].isna()
][
    ['id_pedido', 'pais', 'precio_unitario']
].copy()

# Registros donde sí conocemos el producto
con_producto = orders[
    orders['nombre_producto'].notna()
][
    ['pais', 'precio_unitario', 'nombre_producto']
].copy()

# Buscar coincidencias exactas de país + precio
coincidencias_precio = (
    sin_producto
    .merge(
        con_producto,
        on=['pais', 'precio_unitario'],
        how='left'
    )
)

# Resumir posibles productos por pedido
posibles_productos = (
    coincidencias_precio
    .dropna(subset=['nombre_producto'])
    .groupby('id_pedido')['nombre_producto']
    .agg(
        productos_posibles=lambda x: list(x.unique()),
        cantidad_productos='nunique'
    )
)

display(posibles_productos)

print(
    "Pedidos con alguna coincidencia exacta:",
    posibles_productos.shape[0]
)

print(
    "Pedidos con un único producto posible:",
    (posibles_productos['cantidad_productos'] == 1).sum()
)

,productos_posibles,cantidad_productos
id_pedido,,
order_54,[Vacuum-Pro-Black],1
order_57,[Vacuum-Pro-Black],1
order_69,[Sneakers-Urban-42],1
order_70,[Vacuum-Pro-Black],1


Pedidos con alguna coincidencia exacta: 4
Pedidos con un único producto posible: 4


In [ ]:
# Completar productos y categorías que no pudieron recuperarse
orders['nombre_producto'] = (
    orders['nombre_producto']
    .fillna('Desconocido')
)

orders['categoria_producto'] = (
    orders['categoria_producto']
    .fillna('Desconocido')
)

# Validar
print(
    "Productos faltantes:",
    orders['nombre_producto'].isna().sum()
)

print(
    "Categorías faltantes:",
    orders['categoria_producto'].isna().sum()
)

print("\nDistribución de productos:")
print(orders['nombre_producto'].value_counts())

print("\nDistribución de categorías:")
print(orders['categoria_producto'].value_counts())

Productos faltantes: 0
Categorías faltantes: 0

Distribución de productos:
Blender-XL-Red          4179
Jacket-Winter-M         4176
Vacuum-Pro-Black        4176
Sneakers-Urban-42       4148
Laptop-Gaming-16GB      2782
Tablet-Standard-64GB    2769
Phone-Pro-128GB         2740
Desconocido               30
Name: nombre_producto, dtype: int64

Distribución de categorías:
Hogar          8355
Moda           8324
Electronica    8279
Desconocido      30
Electrónica      12
Name: categoria_producto, dtype: int64


In [ ]:
# Trabajar solo con pedidos que tienen los datos numéricos completos
orders_completos = orders.dropna(
    subset=['cantidad', 'precio_unitario', 'monto_descuento']
)

# Revisar cantidades habituales
print("Distribución de cantidad:")
print(
    orders_completos['cantidad']
    .value_counts()
    .sort_index()
)

print("\nDistribución de descuentos:")
print(
    orders_completos['monto_descuento']
    .value_counts()
    .sort_index()
)

Distribución de cantidad:
-2.0            1
-1.0            3
 1.0        12345
 2.0        12591
 10000.0        6
 20000.0        4
Name: cantidad, dtype: int64

Distribución de descuentos:
0.0     12498
5.0      4920
10.0     5048
15.0     2484
Name: monto_descuento, dtype: int64


In [ ]:
# Distribución considerando únicamente cantidades plausibles
pedidos_normales = orders_completos[
    orders_completos['cantidad'].isin([1, 2])
]

print("Registros con cantidad 1 o 2:", len(pedidos_normales))

print("\nDistribución de cantidad:")
print(pedidos_normales['cantidad'].value_counts().sort_index())

print("\nDistribución de descuentos:")
print(pedidos_normales['monto_descuento'].value_counts().sort_index())

print("\nCombinaciones cantidad / descuento:")
display(
    pd.crosstab(
        pedidos_normales['cantidad'],
        pedidos_normales['monto_descuento']
    )
)

Registros con cantidad 1 o 2: 24936

Distribución de cantidad:
1.0    12345
2.0    12591
Name: cantidad, dtype: int64

Distribución de descuentos:
0.0     12487
5.0      4918
10.0     5047
15.0     2484
Name: monto_descuento, dtype: int64

Combinaciones cantidad / descuento:


monto_descuento,0.0,5.0,10.0,15.0
cantidad,,,,
1.0,6150,2430,2509,1256
2.0,6337,2488,2538,1228


In [ ]:
# Separar cantidades anómalas
cantidades_anomalas = orders[
    orders['cantidad'].notna()
    & ~orders['cantidad'].isin([1, 2])
].copy()

display(
    cantidades_anomalas[
        [
            'id_pedido',
            'id_usuario',
            'fecha_hora_pedido',
            'pais',
            'nombre_producto',
            'categoria_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ].sort_values('cantidad')
)

print(
    "\nCantidad de registros anómalos:",
    len(cantidades_anomalas)
)

,id_pedido,id_usuario,fecha_hora_pedido,pais,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,Colombia,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,Mexico,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,Colombia,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,Colombia,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53
3521,order_3521,user_5812,2025-02-03,Mexico,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.00
3522,order_3522,user_3575,2025-03-29,Argentina,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.00
3586,order_3586,user_3380,2025-02-03,Mexico,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.00
3643,order_3643,user_4440,2025-01-07,Colombia,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.00
3689,order_3689,user_6566,2025-06-16,Mexico,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.00
3748,order_3748,user_7096,2025-02-23,Mexico,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.00



Cantidad de registros anómalos: 14


In [ ]:
# Probar hipótesis de error de escala en cantidades extremas
extremos = orders[
    orders['cantidad'].isin([10000, 20000])
].copy()

extremos['cantidad_corregida'] = extremos['cantidad'] / 10000

extremos['monto_con_cantidad_corregida'] = (
    extremos['cantidad_corregida']
    * extremos['precio_unitario']
    - extremos['monto_descuento']
).round(2)

display(
    extremos[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'cantidad_corregida',
            'precio_unitario',
            'monto_descuento',
            'monto_total',
            'monto_con_cantidad_corregida'
        ]
    ]
)

,id_pedido,nombre_producto,cantidad,cantidad_corregida,precio_unitario,monto_descuento,monto_total,monto_con_cantidad_corregida
3521,order_3521,Laptop-Gaming-16GB,10000.0,1.0,43.14,0.0,431400.0,43.14
3522,order_3522,Laptop-Gaming-16GB,10000.0,1.0,280.55,0.0,2805500.0,280.55
3586,order_3586,Laptop-Gaming-16GB,10000.0,1.0,490.35,0.0,4903500.0,490.35
3643,order_3643,Laptop-Gaming-16GB,10000.0,1.0,238.15,0.0,2381500.0,238.15
3656,order_3656,Laptop-Gaming-16GB,20000.0,2.0,297.66,0.0,5953200.0,595.32
3668,order_3668,Laptop-Gaming-16GB,20000.0,2.0,348.31,0.0,6966200.0,696.62
3689,order_3689,Laptop-Gaming-16GB,10000.0,1.0,87.69,0.0,876900.0,87.69
3722,order_3722,Laptop-Gaming-16GB,20000.0,2.0,442.01,0.0,8840200.0,884.02
3726,order_3726,Laptop-Gaming-16GB,20000.0,2.0,290.85,0.0,5817000.0,581.70
3748,order_3748,Laptop-Gaming-16GB,10000.0,1.0,336.93,0.0,3369300.0,336.93


In [ ]:
# Identificar únicamente las cantidades extremas
mask_extremos = orders['cantidad'].isin([10000, 20000])

print("Registros a corregir:", mask_extremos.sum())

# Corregir error de escala en cantidad
orders.loc[mask_extremos, 'cantidad'] = (
    orders.loc[mask_extremos, 'cantidad'] / 10000
)

# Recalcular monto_total con la cantidad corregida
orders.loc[mask_extremos, 'monto_total'] = (
    orders.loc[mask_extremos, 'cantidad']
    * orders.loc[mask_extremos, 'precio_unitario']
    - orders.loc[mask_extremos, 'monto_descuento']
).round(2)

Registros a corregir: 10


In [ ]:
# Validar corrección
print("Cantidades 10000:", (orders['cantidad'] == 10000).sum())
print("Cantidades 20000:", (orders['cantidad'] == 20000).sum())

print("\nDistribución de cantidades:")
print(orders['cantidad'].value_counts(dropna=False).sort_index())

# Mostrar registros corregidos
display(
    orders.loc[
        mask_extremos,
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

Cantidades 10000: 0
Cantidades 20000: 0

Distribución de cantidades:
-2.0        1
-1.0        3
 1.0    12351
 2.0    12595
 NaN       50
Name: cantidad, dtype: int64


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,Laptop-Gaming-16GB,1.0,43.14,0.0,43.14
3522,order_3522,Laptop-Gaming-16GB,1.0,280.55,0.0,280.55
3586,order_3586,Laptop-Gaming-16GB,1.0,490.35,0.0,490.35
3643,order_3643,Laptop-Gaming-16GB,1.0,238.15,0.0,238.15
3656,order_3656,Laptop-Gaming-16GB,2.0,297.66,0.0,595.32
3668,order_3668,Laptop-Gaming-16GB,2.0,348.31,0.0,696.62
3689,order_3689,Laptop-Gaming-16GB,1.0,87.69,0.0,87.69
3722,order_3722,Laptop-Gaming-16GB,2.0,442.01,0.0,884.02
3726,order_3726,Laptop-Gaming-16GB,2.0,290.85,0.0,581.70
3748,order_3748,Laptop-Gaming-16GB,1.0,336.93,0.0,336.93


In [ ]:
# Identificar pedidos con cantidades negativas
negativos = orders[orders['cantidad'] < 0]

usuarios_negativos = negativos['id_usuario'].unique()

# Revisar historial completo de esos usuarios
historial_negativos = (
    orders[orders['id_usuario'].isin(usuarios_negativos)]
    .sort_values(['id_usuario', 'fecha_hora_pedido'])
)

display(
    historial_negativos[
        [
            'id_pedido',
            'id_usuario',
            'fecha_hora_pedido',
            'pais',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

,id_pedido,id_usuario,fecha_hora_pedido,pais,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
8231,order_8231,user_1087,2025-03-04,Mexico,Sneakers-Urban-42,1.0,73.92,0.0,73.92
3862,order_3862,user_1087,2025-04-21,Mexico,Laptop-Gaming-16GB,1.0,413.39,10.0,403.39
267,order_267,user_1087,2025-05-07,Mexico,Phone-Pro-128GB,-1.0,43.50,5.0,-38.50
24243,order_24243,user_1087,2025-05-27,Mexico,Blender-XL-Red,2.0,251.30,0.0,502.60
24216,order_24216,user_1087,2025-06-06,Mexico,Vacuum-Pro-Black,1.0,352.83,15.0,337.83
13314,order_13314,user_1087,2025-06-23,Mexico,Laptop-Gaming-16GB,2.0,392.18,0.0,784.37
7353,order_7353,user_3323,2025-01-06,Colombia,Tablet-Standard-64GB,2.0,234.67,5.0,464.35
7742,order_7742,user_3323,2025-01-15,Colombia,Phone-Pro-128GB,1.0,454.42,5.0,449.42
269,order_269,user_3323,2025-05-25,Colombia,Phone-Pro-128GB,-1.0,423.53,0.0,-423.53
266,order_266,user_7011,2025-03-13,Colombia,Phone-Pro-128GB,-2.0,101.31,10.0,-192.62


In [ ]:
# Identificar transacciones con cantidad negativa
orders['transaccion_negativa'] = orders['cantidad'] < 0

print("Transacciones negativas:", orders['transaccion_negativa'].sum())

display(
    orders.loc[
        orders['transaccion_negativa'],
        [
            'id_pedido',
            'id_usuario',
            'fecha_hora_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

Transacciones negativas: 4


,id_pedido,id_usuario,fecha_hora_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,Phone-Pro-128GB,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,Phone-Pro-128GB,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,Phone-Pro-128GB,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,Phone-Pro-128GB,-1.0,423.53,0.0,-423.53


**Negative quantities:** Four transactions contained negative quantities. A review of user histories revealed purchases of the same products before or after some of these records, suggesting possible cancellations, returns, or reversals. However, the dataset did not include a field confirming their nature. The original negative values were therefore retained and flagged, avoiding the risk of treating possible reversals as positive sales.


In [ ]:
# Registros con información numérica faltante
numericos_faltantes = orders[
    orders[['cantidad', 'precio_unitario', 'monto_descuento']]
    .isna()
    .any(axis=1)
].copy()

print("Registros con datos numéricos faltantes:", len(numericos_faltantes))

display(
    numericos_faltantes[
        [
            'id_pedido',
            'id_usuario',
            'fecha_hora_pedido',
            'pais',
            'nombre_producto',
            'categoria_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ].head(10)
)

Registros con datos numéricos faltantes: 50


,id_pedido,id_usuario,fecha_hora_pedido,pais,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,Sneakers-Urban-42,Moda,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,Sneakers-Urban-42,Moda,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,Laptop-Gaming-16GB,Electrónica,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,Phone-Pro-128GB,Electrónica,NaN,NaN,NaN,432.39
79,order_79,user_6438,2025-03-22,Colombia,Sneakers-Urban-42,Moda,NaN,NaN,NaN,527.15
80,order_80,user_6790,2025-01-03,Mexico,Vacuum-Pro-Black,Hogar,NaN,NaN,NaN,711.18
81,order_81,user_232,2025-05-16,Mexico,Tablet-Standard-64GB,Electrónica,NaN,NaN,NaN,41.08
82,order_82,user_5665,2025-02-07,Colombia,Sneakers-Urban-42,Moda,NaN,NaN,NaN,431.55
83,order_83,user_5225,2025-04-25,Argentina,Sneakers-Urban-42,Moda,NaN,NaN,NaN,279.35


In [ ]:
# Posibles cantidades y descuentos observados en el dataset
cantidades_posibles = [1, 2]
descuentos_posibles = [0, 5, 10, 15]

resultados = []

for idx, row in numericos_faltantes.iterrows():
    for cantidad in cantidades_posibles:
        for descuento in descuentos_posibles:

            # Despejar precio unitario:
            # monto_total = cantidad * precio_unitario - descuento
            precio_estimado = (
                row['monto_total'] + descuento
            ) / cantidad

            resultados.append({
                'id_pedido': row['id_pedido'],
                'nombre_producto': row['nombre_producto'],
                'monto_total': row['monto_total'],
                'cantidad_posible': cantidad,
                'descuento_posible': descuento,
                'precio_estimado': round(precio_estimado, 2)
            })

posibles_50 = pd.DataFrame(resultados)

display(posibles_50.head(16))

,id_pedido,nombre_producto,monto_total,cantidad_posible,descuento_posible,precio_estimado
0,order_74,Sneakers-Urban-42,595.85,1,0,595.85
1,order_74,Sneakers-Urban-42,595.85,1,5,600.85
2,order_74,Sneakers-Urban-42,595.85,1,10,605.85
3,order_74,Sneakers-Urban-42,595.85,1,15,610.85
4,order_74,Sneakers-Urban-42,595.85,2,0,297.93
5,order_74,Sneakers-Urban-42,595.85,2,5,300.43
6,order_74,Sneakers-Urban-42,595.85,2,10,302.93
7,order_74,Sneakers-Urban-42,595.85,2,15,305.43
8,order_75,Sneakers-Urban-42,458.15,1,0,458.15
9,order_75,Sneakers-Urban-42,458.15,1,5,463.15


In [ ]:
# Registros completos y con cantidades normales
referencia = orders[
    orders['cantidad'].isin([1, 2]) &
    orders['precio_unitario'].notna() &
    orders['monto_descuento'].notna() &
    orders['nombre_producto'].notna()
].copy()

# Combinaciones observadas por producto
combinaciones_producto = (
    referencia
    .groupby(
        ['nombre_producto', 'cantidad', 'monto_descuento']
    )
    .size()
    .reset_index(name='frecuencia')
    .sort_values(
        ['nombre_producto', 'cantidad', 'monto_descuento']
    )
)

display(combinaciones_producto)

,nombre_producto,cantidad,monto_descuento,frecuencia
0,Blender-XL-Red,1.0,0.0,1033
1,Blender-XL-Red,1.0,5.0,396
2,Blender-XL-Red,1.0,10.0,407
3,Blender-XL-Red,1.0,15.0,237
4,Blender-XL-Red,2.0,0.0,1029
...,...,...,...,...
59,Vacuum-Pro-Black,1.0,15.0,211
60,Vacuum-Pro-Black,2.0,0.0,1050
61,Vacuum-Pro-Black,2.0,5.0,415
62,Vacuum-Pro-Black,2.0,10.0,439


In [ ]:
# Analizar precio por producto y país
precio_producto_pais = (
    referencia
    .groupby(['nombre_producto', 'pais'])
    ['precio_unitario']
    .agg(
        pedidos='count',
        precio_min='min',
        precio_mediano='median',
        precio_promedio='mean',
        precio_max='max'
    )
    .round(2)
)

display(precio_producto_pais)

pedidos  precio_min  precio_mediano  \
nombre_producto      pais                                               
Blender-XL-Red       Argentina       1329       20.36          260.84   
                     Colombia        1400       20.06          259.63   
                     Desconocido        4       73.39          109.26   
                     Mexico          1443       20.13          249.61   
Desconocido          Argentina          9       69.82          319.85   
                     Colombia           7       56.50          234.24   
                     Mexico            14       26.86          310.47   
Jacket-Winter-M      Argentina       1353       20.60          270.18   
                     Colombia        1447       20.06          257.00   
                     Desconocido        2      119.20          219.39   
                     Mexico          1364       20.06          253.13   
Laptop-Gaming-16GB   Argentina        916       20.30          277.71   
                     Colombia         930       20.27          245.82   
                     Desconocido        3      302.95          445.13   
                     Mexico           929       20.24          264.29   
Phone-Pro-128GB      Argentina        868       20.40          262.62   
                     Colombia         940       20.08          260.12   
                     Desconocido        3       40.10          125.45   
                     Mexico           922       20.37          262.20   
Sneakers-Urban-42    Argentina       1386       20.11          251.23   
                     Colombia        1347       20.40          261.16   
                     Desconocido        1      467.46          467.46   
                     Mexico          1395       20.29          258.87   
Tablet-Standard-64GB Argentina        916       20.03          240.79   
                     Colombia         910       20.60          265.52   
                     Desconocido        2       93.03          234.52   
                     Mexico           936       20.42          254.84   
Vacuum-Pro-Black     Argentina       1357       20.87          267.33   
                     Colombia        1387       20.37          255.59   
                     Desconocido        4      313.39          380.53   
                     Mexico          1422       20.64          263.90   

                                  precio_promedio  precio_max  
nombre_producto      pais                                      
Blender-XL-Red       Argentina             262.36      499.27  
                     Colombia              260.06      499.93  
                     Desconocido           174.69      406.84  
                     Mexico                255.57      499.94  
Desconocido          Argentina             271.66      437.98  
                     Colombia              259.46      476.78  
                     Mexico                262.58      495.38  
Jacket-Winter-M      Argentina             262.17      499.86  
                     Colombia              256.43      499.93  
                     Desconocido           219.39      319.58  
                     Mexico                261.13      499.79  
Laptop-Gaming-16GB   Argentina             267.97      499.95  
                     Colombia              252.74      499.78  
                     Desconocido           412.84      490.44  
                     Mexico                262.98      499.26  
Phone-Pro-128GB      Argentina             261.64      499.67  
                     Colombia              261.00      499.69  
                     Desconocido           159.25      312.19  
                     Mexico                256.37      499.86  
Sneakers-Urban-42    Argentina             255.20      499.89  
                     Colombia              259.77      499.91  
                     Desconocido           467.46      467.46  
                     Mexico                256.66      499.68  
Tablet-Standard-64GB Arg

In [ ]:
# Resumen general del precio por país
precio_pais = (
    referencia
    .groupby('pais')['precio_unitario']
    .agg(
        pedidos='count',
        precio_min='min',
        precio_mediano='median',
        precio_promedio='mean',
        precio_max='max'
    )
    .round(2)
)

display(precio_pais)

,pedidos,precio_min,precio_mediano,precio_promedio,precio_max
pais,,,,,
Argentina,8134,20.03,260.99,260.12,499.96
Colombia,8368,20.06,257.74,258.71,499.93
Desconocido,19,40.10,313.39,277.39,490.44
Mexico,8425,20.06,256.96,259.26,499.94


In [ ]:
# Analizar comportamiento del precio a través del tiempo
referencia = referencia.copy()

referencia['mes'] = (
    referencia['fecha_hora_pedido']
    .dt.to_period('M')
)

precio_mes = (
    referencia
    .groupby('mes')['precio_unitario']
    .agg(
        pedidos='count',
        precio_mediano='median',
        precio_promedio='mean'
    )
    .round(2)
)

display(precio_mes)

,pedidos,precio_mediano,precio_promedio
mes,,,
2025-01,4318,261.86,259.34
2025-02,3747,253.07,257.10
2025-03,4257,258.30,260.15
2025-04,4215,254.78,257.21
2025-05,4243,266.57,261.07
2025-06,4166,259.48,261.12


In [ ]:
# Precios reales observados por producto
precios_reales = (
    referencia[
        ['nombre_producto', 'precio_unitario']
    ]
    .drop_duplicates()
    .copy()
)

# Redondear a 2 decimales para comparar
precios_reales['precio_unitario'] = (
    precios_reales['precio_unitario'].round(2)
)

posibles_50['precio_estimado'] = (
    posibles_50['precio_estimado'].round(2)
)

# Buscar coincidencias entre precio estimado y precio real
coincidencias = posibles_50.merge(
    precios_reales,
    left_on=['nombre_producto', 'precio_estimado'],
    right_on=['nombre_producto', 'precio_unitario'],
    how='inner'
)

display(
    coincidencias[
        [
            'id_pedido',
            'nombre_producto',
            'monto_total',
            'cantidad_posible',
            'descuento_posible',
            'precio_estimado'
        ]
    ]
)

,id_pedido,nombre_producto,monto_total,cantidad_posible,descuento_posible,precio_estimado
0,order_76,Laptop-Gaming-16GB,319.75,1,5,324.75
1,order_79,Sneakers-Urban-42,527.15,2,10,268.57
2,order_82,Sneakers-Urban-42,431.55,1,15,446.55
3,order_84,Laptop-Gaming-16GB,785.89,2,10,397.94
4,order_85,Jacket-Winter-M,576.13,2,10,293.06
5,order_86,Tablet-Standard-64GB,49.80,1,5,54.80
6,order_88,Blender-XL-Red,856.06,2,15,435.53
7,order_89,Vacuum-Pro-Black,222.21,1,10,232.21
8,order_89,Vacuum-Pro-Black,222.21,1,15,237.21
9,order_89,Vacuum-Pro-Black,222.21,2,10,116.11


In [ ]:
soluciones_por_pedido = (
    coincidencias
    .groupby('id_pedido')
    .size()
    .rename('cantidad_soluciones')
)

resumen_soluciones = (
    numericos_faltantes[
        ['id_pedido', 'nombre_producto', 'monto_total']
    ]
    .merge(
        soluciones_por_pedido,
        on='id_pedido',
        how='left'
    )
)

resumen_soluciones['cantidad_soluciones'] = (
    resumen_soluciones['cantidad_soluciones']
    .fillna(0)
    .astype(int)
)

print("Distribución del número de soluciones:")
print(
    resumen_soluciones['cantidad_soluciones']
    .value_counts()
    .sort_index()
)

display(resumen_soluciones)

Distribución del número de soluciones:
0    29
1    19
3     2
Name: cantidad_soluciones, dtype: int64


,id_pedido,nombre_producto,monto_total,cantidad_soluciones
0,order_74,Sneakers-Urban-42,595.85,0
1,order_75,Sneakers-Urban-42,458.15,0
2,order_76,Laptop-Gaming-16GB,319.75,1
3,order_77,Vacuum-Pro-Black,227.55,0
4,order_78,Phone-Pro-128GB,432.39,0
5,order_79,Sneakers-Urban-42,527.15,1
6,order_80,Vacuum-Pro-Black,711.18,0
7,order_81,Tablet-Standard-64GB,41.08,0
8,order_82,Sneakers-Urban-42,431.55,1
9,order_83,Sneakers-Urban-42,279.35,0


In [ ]:
# Quedarnos solamente con pedidos que tienen una única solución
pedidos_unicos = resumen_soluciones.loc[
    resumen_soluciones['cantidad_soluciones'] == 1,
    'id_pedido'
]

soluciones_unicas = coincidencias[
    coincidencias['id_pedido'].isin(pedidos_unicos)
].copy()

# Preparar registros completos para comparar
referencia_combinaciones = referencia[
    [
        'nombre_producto',
        'cantidad',
        'monto_descuento',
        'precio_unitario'
    ]
].copy()

# Redondear precio
referencia_combinaciones['precio_unitario'] = (
    referencia_combinaciones['precio_unitario'].round(2)
)

# Contar cuántas veces aparece cada combinación completa
frecuencia_combinacion = (
    referencia_combinaciones
    .groupby(
        [
            'nombre_producto',
            'cantidad',
            'monto_descuento',
            'precio_unitario'
        ]
    )
    .size()
    .reset_index(name='frecuencia')
)

# Comparar las soluciones inferidas con combinaciones reales
validacion_unicas = soluciones_unicas.merge(
    frecuencia_combinacion,
    left_on=[
        'nombre_producto',
        'cantidad_posible',
        'descuento_posible',
        'precio_estimado'
    ],
    right_on=[
        'nombre_producto',
        'cantidad',
        'monto_descuento',
        'precio_unitario'
    ],
    how='left'
)

validacion_unicas['frecuencia'] = (
    validacion_unicas['frecuencia']
    .fillna(0)
    .astype(int)
)

display(
    validacion_unicas[
        [
            'id_pedido',
            'nombre_producto',
            'monto_total',
            'cantidad_posible',
            'descuento_posible',
            'precio_estimado',
            'frecuencia'
        ]
    ]
)

,id_pedido,nombre_producto,monto_total,cantidad_posible,descuento_posible,precio_estimado,frecuencia
0,order_76,Laptop-Gaming-16GB,319.75,1,5,324.75,0
1,order_79,Sneakers-Urban-42,527.15,2,10,268.57,0
2,order_82,Sneakers-Urban-42,431.55,1,15,446.55,0
3,order_84,Laptop-Gaming-16GB,785.89,2,10,397.94,0
4,order_85,Jacket-Winter-M,576.13,2,10,293.06,0
5,order_86,Tablet-Standard-64GB,49.80,1,5,54.80,1
6,order_88,Blender-XL-Red,856.06,2,15,435.53,0
7,order_91,Sneakers-Urban-42,474.59,1,0,474.59,0
8,order_93,Blender-XL-Red,359.47,2,5,182.24,0
9,order_95,Jacket-Winter-M,444.51,2,0,222.25,0


In [ ]:
# Frecuencia de cada precio unitario en los registros completos
frecuencia_precios = (
    referencia
    .groupby(['nombre_producto', 'precio_unitario'])
    .size()
    .reset_index(name='frecuencia')
)

print("Total de combinaciones producto/precio:", len(frecuencia_precios))

print(
    "\nCombinaciones que aparecen una sola vez:",
    (frecuencia_precios['frecuencia'] == 1).sum()
)

print(
    "Combinaciones que aparecen más de una vez:",
    (frecuencia_precios['frecuencia'] > 1).sum()
)

print(
    "\nPorcentaje de precios únicos:",
    round(
        (frecuencia_precios['frecuencia'] == 1).mean() * 100,
        2
    ),
    "%"
)

Total de combinaciones producto/precio: 24058

Combinaciones que aparecen una sola vez: 23188
Combinaciones que aparecen más de una vez: 870

Porcentaje de precios únicos: 96.38 %


In [ ]:
display(
    frecuencia_precios['frecuencia']
    .value_counts()
    .sort_index()
    .head(10)
)

1    23188
2      852
3       18
Name: frecuencia, dtype: int64

**Handling missing numerical values:**

Fifty records had missing values in `cantidad`, `precio_unitario`, and `monto_descuento`, although `monto_total` was available. Potential reconstruction was assessed using observed quantities (1–2 units), discounts (0, 5, 10, and 15), and historical prices by product, country, and period.

However, 96.38% of product–price combinations occurred only once, indicating substantial unit-price variability. Because multiple combinations could explain the same total and the original values could not be reliably identified, these fields were left missing rather than imputed without supporting evidence. The records were retained because `monto_total` remained available for revenue analysis.


In [ ]:
# Validación final de los datasets limpios

# Eliminar columnas auxiliares creadas durante la limpieza
columnas_auxiliares = ['monto_calculado', 'diferencia_monto']

orders.drop(
    columns=[col for col in columnas_auxiliares if col in orders.columns],
    inplace=True
)

print("=== ORDERS ===")
print("Dimensiones:", orders.shape)
print("\nValores faltantes:")
print(orders.isna().sum())

print("\nDuplicados exactos:", orders.duplicated().sum())
print("IDs de pedido duplicados:", orders['id_pedido'].duplicated().sum())

print("\nDistribución de cantidad:")
print(orders['cantidad'].value_counts(dropna=False).sort_index())

print("\n=== CATALOG ===")
print("Dimensiones:", catalog.shape)
print("Valores faltantes:", catalog.isna().sum().sum())
print("Duplicados exactos:", catalog.duplicated().sum())
print(
    "Productos duplicados:",
    catalog['nombre_producto'].duplicated().sum()
)

print("\n=== MARKETING ===")
print("Dimensiones:", marketing.shape)
print("\nValores faltantes:")
print(marketing.isna().sum())

print("\nDuplicados exactos:", marketing.duplicated().sum())

=== ORDERS ===
Dimensiones: (25000, 13)

Valores faltantes:
id_pedido                0
id_usuario               0
fecha_hora_pedido        0
pais                     0
dispositivo              0
fuente_referencia        0
nombre_producto          0
categoria_producto       0
cantidad                50
precio_unitario         50
monto_descuento         50
monto_total              0
transaccion_negativa     0
dtype: int64

Duplicados exactos: 0
IDs de pedido duplicados: 0

Distribución de cantidad:
-2.0        1
-1.0        3
 1.0    12351
 2.0    12595
 NaN       50
Name: cantidad, dtype: int64

=== CATALOG ===
Dimensiones: (7, 4)
Valores faltantes: 0
Duplicados exactos: 0
Productos duplicados: 0

=== MARKETING ===
Dimensiones: (1620, 5)

Valores faltantes:
fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

Duplicados exactos: 0


---

### Exporting Cleaned Data

After completing the data-cleaning and validation process, the cleaned datasets are exported for use in the subsequent analyses and the Power BI reporting stage.


In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print("Datasets exportados correctamente.")

---

## Step 2 | Business Profitability Analysis

### 2.1 Key Performance Indicators (KPIs)

**Objective:** Evaluate revenue, known product costs, marketing expenditure, estimated profitability, and sales behavior using the `orders`, `catalog`, and `marketing_spend` datasets.

**Business performance:** Assess total revenue, known costs, marketing investment, and estimated profit. Costs are only calculated when both quantity and unit cost are available.

**Sales behavior:** Examine average order value, average item quantity, best-selling products by units and by order frequency, and marketing expenditure by channel.

The metrics below are accompanied by validation checks and interpretation notes.


In [ ]:
# 1. Total revenue

revenue_total = orders['monto_total'].sum()

# Separate non-negative and negative transactions
revenue_positivo = orders.loc[
    orders['transaccion_negativa'] == 0,
    'monto_total'
].sum()

revenue_negativo = orders.loc[
    orders['transaccion_negativa'] == 1,
    'monto_total'
].sum()

print(f"Total revenue (including negative transactions): ${revenue_total:,.2f}")
print(f"Revenue from non-negative transactions: ${revenue_positivo:,.2f}")
print(f"Impact of negative transactions: ${revenue_negativo:,.2f}")


**Total revenue:** Net revenue for the period was **9,646,996.73 USD**. Four transactions with negative amounts were retained because they may represent cancellations, returns, or reversals; their exact nature could not be confirmed. Their combined impact was **-1,147.30 USD**, approximately **0.012%** of revenue, with no material effect on the overall result.


In [ ]:
# Join orders with unit costs from the product catalog
orders_cost = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

print("Dimensions:", orders_cost.shape)

print("\nOrders without a unit cost:")
print(orders_cost['costo_unitario'].isna().sum())

print("\nOrders without a quantity:")
print(orders_cost['cantidad'].isna().sum())

In [ ]:
# Check the availability of fields needed to calculate costs
estado_costos = pd.crosstab(
    orders_cost['cantidad'].isna(),
    orders_cost['costo_unitario'].isna()
)

estado_costos.index = ['Quantity available', 'Quantity missing']
estado_costos.columns = ['Cost available', 'Cost missing']

display(estado_costos)

In [ ]:
# Identify orders whose costs cannot be calculated
sin_costo_calculable = orders_cost[
    orders_cost['cantidad'].isna()
    | orders_cost['costo_unitario'].isna()
]

print("Orders without calculable costs:", len(sin_costo_calculable))

print(
    "Associated revenue:",
    f"${sin_costo_calculable['monto_total'].sum():,.2f}"
)

print(
    "Share of total revenue:",
    f"{sin_costo_calculable['monto_total'].sum() / revenue_total * 100:.2f}%"
)

In [ ]:
# Calculate cost per order
orders_cost['costo_total_pedido'] = (
    orders_cost['cantidad']
    * orders_cost['costo_unitario']
)

# Known total cost
costo_total = orders_cost['costo_total_pedido'].sum()

# Coverage
pedidos_con_costo = orders_cost['costo_total_pedido'].notna().sum()
cobertura_costos = pedidos_con_costo / len(orders_cost) * 100

print(f"Known total cost: ${costo_total:,.2f}")
print(f"Orders included in calculation: {pedidos_con_costo:,}")
print(f"Order coverage: {cobertura_costos:.2f}%")

In [ ]:
# Effect of negative transactions on costs
costo_transacciones_negativas = orders_cost.loc[
    orders_cost['transaccion_negativa'] == 1,
    'costo_total_pedido'
].sum()

costo_sin_negativas = orders_cost.loc[
    orders_cost['transaccion_negativa'] == 0,
    'costo_total_pedido'
].sum()

print(f"Cost including negative transactions: ${costo_total:,.2f}")
print(f"Cost excluding negative transactions: ${costo_sin_negativas:,.2f}")
print(
    f"Impact of negative transactions: "
    f"${costo_transacciones_negativas:,.2f}"
)

**Known product costs:** Calculated product costs totaled **3,832,747.93 USD**, covering **24,920 orders (99.68%)**. The remaining 80 orders lacked sufficient information to determine their costs and accounted for approximately **0.35% of revenue**; no unsupported values were imputed. The four negative transactions contributed **-50.60 USD** to calculated costs and were handled consistently with revenue.


In [ ]:
# 3. Total marketing investment

gasto_marketing = marketing['gasto'].sum()

print(f"Total marketing investment: ${gasto_marketing:,.2f}")

In [ ]:
# Validate expenditure values

print("Minimum spend:", marketing['gasto'].min())
print("Maximum spend:", marketing['gasto'].max())
print("Negative spend entries:", (marketing['gasto'] < 0).sum())
print("Zero spend entries:", (marketing['gasto'] == 0).sum())

**Marketing investment:** Total recorded marketing expenditure was **2,871,843.53 USD**. No zero or negative expenditure entries were identified, so the recorded total was used in the profitability calculation.


In [ ]:
# 4. Calculate estimated profit

profit = revenue_total - costo_total - gasto_marketing

print(f"Revenue: ${revenue_total:,.2f}")
print(f"Known cost: ${costo_total:,.2f}")
print(f"Marketing spend: ${gasto_marketing:,.2f}")
print(f"Estimated profit: ${profit:,.2f}")

In [ ]:
# Estimated profit margin relative to revenue
margen_profit = (profit / revenue_total) * 100

print(f"Estimated profit margin: {margen_profit:.2f}%")

**Estimated business profitability:** Revenue reached **9,646,996.73 USD**. After deducting **3,832,747.93 USD** in known product costs and **2,871,843.53 USD** in marketing expenditure, estimated profit was **2,942,405.27 USD**, equivalent to an estimated **30.50%** margin on revenue.

This indicates a positive result under the modeled cost assumptions, **not a verified net profit**. Product costs were calculable for 99.68% of orders, and other potential operating expenses were not included.


In [ ]:
# Average order value
ticket_promedio = orders['monto_total'].mean()

print(f"Average order value: {ticket_promedio:,.2f} USD")

# Validate using revenue divided by unique orders
num_ordenes = orders['id_pedido'].nunique()
ticket_validacion = orders['monto_total'].sum() / num_ordenes

print(f"Number of orders: {num_ordenes:,}")
print(f"Average order value (validation): {ticket_validacion:,.2f} USD")

# Compare average order value with and without negative transactions
ticket_con_negativas = orders['monto_total'].mean()

ticket_sin_negativas = (
    orders.loc[~orders['transaccion_negativa'], 'monto_total']
    .mean()
)

print(f"Average order value including negative transactions: {ticket_con_negativas:,.2f} USD")
print(f"Average order value excluding negative transactions: {ticket_sin_negativas:,.2f} USD")
print(f"Difference: {ticket_sin_negativas - ticket_con_negativas:,.2f} USD")

**Average order value:** The average order generated **385.88 USD**. Excluding negative transactions increased the figure to **385.99 USD**, a difference of **0.11 USD** per order. Negative transactions were therefore retained in the primary metric.


In [ ]:
# Average quantity per order
cantidad_promedio = orders['cantidad'].mean()

pedidos_con_cantidad = orders['cantidad'].notna().sum()
cobertura_cantidad = pedidos_con_cantidad / len(orders) * 100

print(f"Average quantity per order: {cantidad_promedio:.2f}")
print(f"Orders included in calculation: {pedidos_con_cantidad:,}")
print(f"Order coverage: {cobertura_cantidad:.2f}%")

# Compare averages including and excluding negative quantities
cantidad_promedio_con_negativas = orders['cantidad'].mean()

cantidad_promedio_sin_negativas = (
    orders.loc[
        orders['cantidad'].notna() & (orders['cantidad'] > 0),
        'cantidad'
    ]
    .mean()
)

print(
    f"Average including negative quantities: "
    f"{cantidad_promedio_con_negativas:.4f}"
)

print(
    f"Average using only positive quantities: "
    f"{cantidad_promedio_sin_negativas:.4f}"
)

print(
    f"Difference: "
    f"{cantidad_promedio_sin_negativas - cantidad_promedio_con_negativas:.4f}"
)

**Average quantity per order:** Orders with recorded quantities contained an average of **1.50 items**. This calculation covers **99.80%** of orders; 50 records with missing quantities were left unimputed. Restricting the calculation to positive quantities changes the average from **1.5044** to **1.5049**, indicating minimal sensitivity to negative quantities. Note that this notebook metric uses the mean of non-missing `cantidad` values.


In [ ]:
# Number of orders by product
ventas_por_pedidos = (
    orders[orders['nombre_producto'] != 'Desconocido']
    .groupby('nombre_producto')
    .agg(
        numero_pedidos=('id_pedido', 'nunique')
    )
    .sort_values('numero_pedidos', ascending=False)
)

display(ventas_por_pedidos)

# Net units sold by product
ventas_por_unidades = (
    orders[
        (orders['nombre_producto'] != 'Desconocido') &
        (orders['cantidad'].notna())
    ]
    .groupby('nombre_producto')
    .agg(
        unidades_netas=('cantidad', 'sum'),
        pedidos_con_cantidad=('id_pedido', 'nunique')
    )
    .sort_values('unidades_netas', ascending=False)
)

display(ventas_por_unidades)

In [ ]:
# Positive units by product
ventas_unidades_positivas = (
    orders[
        (orders['nombre_producto'] != 'Desconocido') &
        (orders['cantidad'] > 0)
    ]
    .groupby('nombre_producto')
    .agg(
        unidades_positivas=('cantidad', 'sum')
    )
    .sort_values('unidades_positivas', ascending=False)
)

display(ventas_unidades_positivas)

**Best-selling products:** **Vacuum-Pro-Black** led by net units sold (**6,284**), while **Blender-XL-Red** appeared in the greatest number of orders (**4,179**). The leader by order frequency is therefore different from the leader by unit volume. Negative transactions did not change the ranking by net units.


In [ ]:
# Total marketing expenditure by channel
gasto_por_canal = (
    marketing
    .groupby('canal', as_index=False)
    .agg(
        gasto_total=('gasto', 'sum')
    )
    .sort_values('gasto_total', ascending=False)
)

display(gasto_por_canal)

# Share of marketing investment by channel
gasto_por_canal['porcentaje_gasto'] = (
    gasto_por_canal['gasto_total'] /
    gasto_por_canal['gasto_total'].sum() * 100
)

display(gasto_por_canal)


**Marketing expenditure by channel:** Total marketing spending of **2,871,843.53 USD** was distributed relatively evenly across three channels: `social` accounted for **976,818.37 USD (34.01%)**, `organic` for **972,650.96 USD (33.87%)**, and `paid_search` for **922,374.20 USD (32.12%)**. These are spending shares, not measures of channel effectiveness or attributed return.


---

## Step 3 | Conversion Funnel Analysis

**Objective:** Examine user activity across the purchasing funnel to identify where the largest decreases in unique-user counts occur.

**Data source:** PostgreSQL `events` table. SQL queries summarize event activity and compare users across the following expected stages:

`first_visit` → `select_item` → `add_to_cart` → `begin_checkout` → `add_payment_info` → `purchase`

**Part 1 — Funnel construction and validation**
- Count events and distinct users at each stage.
- Examine events per session and overlaps between consecutive stages.
- Check whether users have recorded consecutive events in chronological order.

**Part 2 — Conversion analysis**
- Compare distinct-user counts between adjacent stages.
- Identify the largest decrease in user counts.
- Calculate the aggregate `purchase`-to-`first_visit` ratio.

**Methodological note:** Stage-level counts do not require every user to complete each preceding event. Therefore, these ratios are **aggregate stage comparisons**, not verified end-to-end user conversion probabilities.

---


### PostgreSQL access and reproducibility

Steps 3 and 4 query a TripleTen academic PostgreSQL database. Access is restricted and may not be available to external readers. The saved results in this notebook document the analysis; executing those SQL cells requires valid, authorized database credentials and network access.

Before running the database connection cell, set `RAPPIPLUS_DB_PASSWORD` in your **local environment**. Do not paste credentials into the notebook or commit them to GitHub. For an interactive private Jupyter session, you may run `os.environ["RAPPIPLUS_DB_PASSWORD"] = getpass("Database password: ")` in a temporary cell (after importing `os` and `getpass`), and remove that cell before publishing.

The CSV-based sections also require their source files to be available. The included outputs are historical results from the validated academic analysis, not a guarantee of live database connectivity.


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Academic PostgreSQL database; credentials are never stored in this notebook.
password = os.getenv('RAPPIPLUS_DB_PASSWORD')
if not password:
    raise RuntimeError(
        'Database access requires RAPPIPLUS_DB_PASSWORD. '
        'See the PostgreSQL access and reproducibility note above.'
    )

db_config = {
    'user': 'practicum_student',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_url = URL.create(
    'postgresql+psycopg2',
    username=db_config['user'],
    password=password,
    host=db_config['host'],
    port=db_config['port'],
    database=db_config['db']
)
engine = create_engine(connection_url, connect_args={'sslmode': 'require'})


In [ ]:
# Explore the events table
# =========================
query_events = '''
SELECT
    nombre_evento,
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY total_eventos DESC;
'''

eventos_resumen = pd.read_sql(query_events, con=engine)
eventos_resumen

In [ ]:
# PART 1: Funnel totals
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# Review event counts per session

query_sesiones = '''
SELECT
    COUNT(*) AS total_sesiones,
    MIN(eventos_por_sesion) AS minimo_eventos,
    MAX(eventos_por_sesion) AS maximo_eventos,
    AVG(eventos_por_sesion) AS promedio_eventos
FROM (
    SELECT
        id_sesion,
        COUNT(*) AS eventos_por_sesion
    FROM events
    GROUP BY id_sesion
) AS sesiones;
'''

sesiones_resumen = pd.read_sql(query_sesiones, con=engine)
sesiones_resumen

In [ ]:
# Check users appearing in adjacent funnel stages

query_interseccion = '''
WITH usuarios_eventos AS (
    SELECT DISTINCT
        id_usuario,
        nombre_evento
    FROM events
),
etapas AS (
    SELECT
        id_usuario,
        MAX(CASE WHEN nombre_evento = 'first_visit' THEN 1 ELSE 0 END) AS first_visit,
        MAX(CASE WHEN nombre_evento = 'select_item' THEN 1 ELSE 0 END) AS select_item,
        MAX(CASE WHEN nombre_evento = 'add_to_cart' THEN 1 ELSE 0 END) AS add_to_cart,
        MAX(CASE WHEN nombre_evento = 'begin_checkout' THEN 1 ELSE 0 END) AS begin_checkout,
        MAX(CASE WHEN nombre_evento = 'add_payment_info' THEN 1 ELSE 0 END) AS add_payment_info,
        MAX(CASE WHEN nombre_evento = 'purchase' THEN 1 ELSE 0 END) AS purchase
    FROM usuarios_eventos
    GROUP BY id_usuario
)
SELECT
    SUM(first_visit) AS usuarios_first_visit,
    SUM(CASE WHEN first_visit = 1 AND select_item = 1 THEN 1 ELSE 0 END)
        AS first_visit_a_select_item,

    SUM(select_item) AS usuarios_select_item,
    SUM(CASE WHEN select_item = 1 AND add_to_cart = 1 THEN 1 ELSE 0 END)
        AS select_item_a_add_to_cart,

    SUM(add_to_cart) AS usuarios_add_to_cart,
    SUM(CASE WHEN add_to_cart = 1 AND begin_checkout = 1 THEN 1 ELSE 0 END)
        AS add_to_cart_a_checkout,

    SUM(begin_checkout) AS usuarios_checkout,
    SUM(CASE WHEN begin_checkout = 1 AND add_payment_info = 1 THEN 1 ELSE 0 END)
        AS checkout_a_payment,

    SUM(add_payment_info) AS usuarios_payment,
    SUM(CASE WHEN add_payment_info = 1 AND purchase = 1 THEN 1 ELSE 0 END)
        AS payment_a_purchase
FROM etapas;
'''

interseccion = pd.read_sql(query_interseccion, con=engine)
interseccion

In [ ]:
# Validate chronological transitions efficiently

query_secuencia = '''
WITH transiciones AS (
    SELECT DISTINCT
        e1.id_usuario,
        e1.nombre_evento AS etapa_actual,
        e2.nombre_evento AS etapa_siguiente
    FROM events e1
    JOIN events e2
        ON e1.id_usuario = e2.id_usuario
       AND e2.timestamp_evento > e1.timestamp_evento
    WHERE
        (e1.nombre_evento = 'first_visit'
            AND e2.nombre_evento = 'select_item')
        OR
        (e1.nombre_evento = 'select_item'
            AND e2.nombre_evento = 'add_to_cart')
        OR
        (e1.nombre_evento = 'add_to_cart'
            AND e2.nombre_evento = 'begin_checkout')
        OR
        (e1.nombre_evento = 'begin_checkout'
            AND e2.nombre_evento = 'add_payment_info')
        OR
        (e1.nombre_evento = 'add_payment_info'
            AND e2.nombre_evento = 'purchase')
)
SELECT
    etapa_actual,
    etapa_siguiente,
    COUNT(DISTINCT id_usuario) AS usuarios_conversion
FROM transiciones
GROUP BY etapa_actual, etapa_siguiente
ORDER BY CASE etapa_actual
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
END;
'''

secuencia = pd.read_sql(query_secuencia, con=engine)
secuencia


,etapa_actual,etapa_siguiente,usuarios_conversion
0,first_visit,select_item,6714
1,select_item,add_to_cart,6364
2,add_to_cart,begin_checkout,5868
3,begin_checkout,add_payment_info,4414
4,add_payment_info,purchase,3514


### Part 1 | Funnel Construction — Findings

- The funnel contains **7,796 distinct users** with a `first_visit` event and **6,240 distinct users** with a `purchase` event.
- Events are presented in the expected order: `first_visit` → `select_item` → `add_to_cart` → `begin_checkout` → `add_payment_info` → `purchase`.
- Distinct-user groups are not strictly sequential. Some users have repeated events or events without a recorded preceding stage. The overlap and timestamp checks provide additional context but do not establish that every user completed the full funnel in order.


In [ ]:

# PART 2: Aggregate stage ratios
# =====================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS etapa
    FROM events
    GROUP BY nombre_evento
),
conversiones AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (ORDER BY etapa) AS usuarios_etapa_anterior,
        etapa
    FROM funnel
)
SELECT
    nombre_evento,
    usuarios_unicos,
    usuarios_etapa_anterior,
    ROUND(
        usuarios_unicos * 100.0 /
        NULLIF(usuarios_etapa_anterior, 0),
        2
    ) AS conversion_paso
FROM conversiones
ORDER BY etapa;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,usuarios_etapa_anterior,conversion_paso
0,first_visit,7796,NaN,NaN
1,select_item,7582,7796.0,97.26
2,add_to_cart,7634,7582.0,100.69
3,begin_checkout,7208,7634.0,94.42
4,add_payment_info,6250,7208.0,86.71
5,purchase,6240,6250.0,99.84


In [ ]:
conversion_final = (
    conversion.loc[
        conversion['nombre_evento'] == 'purchase',
        'usuarios_unicos'
    ].iloc[0]
    /
    conversion.loc[
        conversion['nombre_evento'] == 'first_visit',
        'usuarios_unicos'
    ].iloc[0]
) * 100

print(f"Aggregate purchase-to-first_visit ratio: {conversion_final:.2f}%")

### Part 2 | Conversion Analysis — Findings

- The largest decrease in distinct-user counts occurs between `begin_checkout` and `add_payment_info`: **7,208** versus **6,250** users, a difference of **958 users** and an aggregate stage ratio of **86.71%**.
- The aggregate `purchase`-to-`first_visit` ratio is **80.04%**.
- The `select_item`-to-`add_to_cart` ratio reaches **100.69%**, indicating that the sets of users recorded at adjacent stages are not nested.

**Interpretation and limitation:** These percentages compare distinct users who generated each event, rather than measuring a confirmed sequential journey for the same users. They should not be described as a fully validated funnel conversion rate. For a stricter funnel, each user would need to be followed through the required stages in chronological order, ideally within a defined session or attribution window.

**Business implication:** The decrease around payment information entry warrants investigation, but these aggregated results alone cannot establish the reason for the difference or confirm actual abandonment at that step.


---

## Step 4 | Cohort Retention Analysis

**Objective:** Assess whether users return and remain active after registration by comparing weekly activity across monthly registration cohorts.

**Source tables:** `users` and `user_activity`.

**Methodology**

1. Assign each user to a cohort based on their **registration month**.
2. Identify distinct users with recorded activity on days **7, 14, and 21** after registration, using `activo = 1`:
   - `retenido_w1`: active users on day 7 (W1)
   - `retenido_w2`: active users on day 14 (W2)
   - `retenido_w3`: active users on day 21 (W3)
3. Calculate retention for each cohort by dividing active users at each checkpoint by the number of users initially registered in that cohort:
   - `semana_1`: W1 retention rate (%)
   - `semana_2`: W2 retention rate (%)
   - `semana_3`: W3 retention rate (%)

Registration dates are explicitly converted to SQL `DATE` values using `CAST(fecha_registro AS DATE)`.

**Interpretation note:** These are activity rates at specific post-registration checkpoints, not a measure of uninterrupted weekly engagement or cumulative user loss.


In [ ]:
# Explore the users table
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
query_users_info = '''
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    MIN(CAST(fecha_registro AS DATE)) AS primera_fecha_registro,
    MAX(CAST(fecha_registro AS DATE)) AS ultima_fecha_registro,
    COUNT(*) FILTER (
        WHERE fecha_registro IS NULL
    ) AS fechas_faltantes
FROM users;
'''

users_info = pd.read_sql(query_users_info, con=engine)
users_info

,total_registros,usuarios_unicos,primera_fecha_registro,ultima_fecha_registro,fechas_faltantes
0,8000,8000,2025-01-01,2025-05-31,0


In [ ]:
# Explore the user_activity table
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Validate the user_activity table structure

query_activity_info = '''
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos,
    MIN(CAST(fecha_actividad AS DATE)) AS primera_actividad,
    MAX(CAST(fecha_actividad AS DATE)) AS ultima_actividad,
    MIN(dias_despues_registro) AS min_dias,
    MAX(dias_despues_registro) AS max_dias,
    COUNT(*) FILTER (WHERE fecha_actividad IS NULL) AS fechas_faltantes,
    COUNT(*) FILTER (WHERE activo IS NULL) AS activo_faltante
FROM user_activity;
'''

activity_info = pd.read_sql(query_activity_info, con=engine)
activity_info

,total_registros,usuarios_unicos,primera_actividad,ultima_actividad,min_dias,max_dias,fechas_faltantes,activo_faltante
0,32000,8000,2025-01-08,2025-06-28,7,28,0,0


In [ ]:
query_activity_values = '''
SELECT
    dias_despues_registro,
    activo,
    COUNT(*) AS registros
FROM user_activity
GROUP BY dias_despues_registro, activo
ORDER BY dias_despues_registro, activo;
'''

activity_values = pd.read_sql(query_activity_values, con=engine)
activity_values

,dias_despues_registro,activo,registros
0,7,0,4640
1,7,1,3360
2,14,0,4645
3,14,1,3355
4,21,0,4650
5,21,1,3350
6,28,0,4750
7,28,1,3250


In [ ]:

# Cohort retention analysis
# ======================

query_cohort_retention_final = '''
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),

actividad AS (
    SELECT
        id_usuario,
        dias_despues_registro,
        activo
    FROM user_activity
),

retencion AS (
    SELECT
        u.cohorte,
        COUNT(DISTINCT u.id_usuario) AS usuarios_iniciales,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7
                 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w1,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14
                 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w2,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21
                 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w3

    FROM usuarios_cohorte u
    LEFT JOIN actividad a
        ON u.id_usuario = a.id_usuario
    GROUP BY u.cohorte
)

SELECT
    cohorte,
    usuarios_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / usuarios_iniciales, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / usuarios_iniciales, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / usuarios_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Run the query
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


In [ ]:
# Average retention across cohorts
promedio_retencion = cohorte_final[
    ['semana_1', 'semana_2', 'semana_3']
].mean().round(2)

print(promedio_retencion)

semana_1    42.01
semana_2    41.96
semana_3    41.92
dtype: float64


### Key Findings | Cohort Retention

- Retention remained relatively stable across the three post-registration checkpoints.
- Across cohorts, the **unweighted mean retention rates** were **42.01% at W1**, **41.96% at W2**, and **41.92% at W3**, with no substantial progressive decline.
- Individual cohorts showed modest differences. The January cohort decreased from **42.84% to 40.32%**, while some cohorts recovered at later checkpoints. The February cohort reached **43.98% at W3**, the highest reported value.
- Overall, approximately **four in ten registered users** were active at each of the first three measured checkpoints.
- Users can be inactive at one checkpoint and active again later. Therefore, changes between W1, W2, and W3 should **not** be interpreted as cumulative churn.

**Business implication:** These results provide a baseline for monitoring early user engagement. Further analysis of user segments and activity patterns would help identify opportunities to encourage repeat engagement.


---

## Step 5 | A/B Testing and Statistical Significance

### Business objective

Evaluate whether a change to the checkout user interface (UI) is associated with a difference in the **purchase conversion rate** between the control and treatment groups.

**Dataset:** `experiment_checkout_ui.csv`. The observed conversion indicator is stored in the `convirtio` column (1 = completed a purchase; 0 = did not complete a purchase).

**Analytical approach:**
1. Inspect the experiment dataset and validate its structure.
2. Define the null and alternative hypotheses.
3. Apply a two-sided Z-test for two independent proportions.
4. Interpret the statistical evidence and its business implications.

---


### Statistical hypotheses

- **H₀ (null hypothesis):** The purchase conversion rates are equal in the control and treatment groups.
- **H₁ (alternative hypothesis):** The purchase conversion rates differ between the control and treatment groups.

**Statistical method:** Two-sided Z-test for two independent proportions.

**Significance level:** α = 0.05.

The test evaluates evidence of a difference, not whether the treatment is guaranteed to improve conversion.


In [ ]:
# Explore the experiment dataset

import pandas as pd

url = 'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'

experiment = pd.read_csv(url)

display(experiment.head())

print("Dataset dimensions:", experiment.shape)

print("\nData types:")
print(experiment.dtypes)

print("\nMissing values:")
print(experiment.isna().sum())

print("\nExact duplicates:", experiment.duplicated().sum())


In [ ]:
# Validate the experiment structure

print("Conversion indicator values:")
print(experiment['convirtio'].value_counts(dropna=False).sort_index())

print("\nVariant distribution:")
print(experiment['variante'].value_counts())

print("\nUnique users:", experiment['id_usuario'].nunique())

# Check whether any user is assigned to more than one variant
variantes_usuario = (
    experiment
    .groupby('id_usuario')['variante']
    .nunique()
)

print(
    "\nUsers assigned to more than one variant:",
    (variantes_usuario > 1).sum()
)


In [ ]:
# Summarize conversion by experiment variant

resumen_experimento = (
    experiment
    .groupby('variante')
    .agg(
        usuarios=('id_usuario', 'nunique'),
        conversiones=('convirtio', 'sum'),
        tasa_conversion=('convirtio', 'mean')
    )
    .reset_index()
)

resumen_experimento['tasa_conversion'] *= 100

display(resumen_experimento.round(2))


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Observed conversions
conversiones = [
    experiment.loc[experiment['variante'] == 'control', 'convirtio'].sum(),
    experiment.loc[experiment['variante'] == 'tratamiento', 'convirtio'].sum()
]

# Sample size in each group
usuarios = [
    (experiment['variante'] == 'control').sum(),
    (experiment['variante'] == 'tratamiento').sum()
]

# Significance level
alpha = 0.05

# Two-sided Z-test for two independent proportions
z_stat, p_value = proportions_ztest(
    count=conversiones,
    nobs=usuarios,
    alternative='two-sided'
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Alpha: {alpha}")

if p_value < alpha:
    print("Reject H0.")
    print("There is evidence of a statistically significant difference.")
else:
    print("Fail to reject H0.")
    print("There is insufficient evidence of a statistically significant difference.")


### A/B Test Results and Interpretation

The observed purchase conversion rate was **15.69%** for the **control** group and **16.29%** for the **treatment** group, a difference of **0.60 percentage points** in favor of the updated checkout UI.

A two-sided Z-test for two independent proportions yielded **Z = -0.8133** and **p = 0.4162**. Because the p-value exceeds the predefined significance level (**α = 0.05**), we **fail to reject the null hypothesis**.

**Conclusion:** The experiment does not provide sufficient statistical evidence that the checkout UI change affected purchase conversion. Although the treatment group had a slightly higher observed conversion rate, the difference is compatible with random sampling variability.

**Business implication:** The observed uplift alone does not justify claiming that the redesigned checkout improved conversion. Any rollout decision should also consider practical impact, experiment design, and additional evidence.


---

## Step 6 | Power BI Dashboards and Business Insights

**Objective:** Communicate sales, product-cost, marketing, and profitability findings through interactive Power BI reports that support commercial decision-making.

**Prepared data:** The cleaned datasets exported in Step 1 were used as inputs:

- `orders_clean.csv`
- `catalog_clean.csv`
- `marketing_clean.csv`

### Data model and reporting approach

The Power BI model connects order records to the product catalog through `nombre_producto` and uses a date dimension (`dim_fecha`) for time-based reporting. Marketing expenditure is related to the date dimension for global-period analysis; it is **not allocated to individual products or categories**.

The report distinguishes **gross profit**, calculated from revenue and known product costs, from **estimated overall profit**, which additionally deducts global marketing expenditure. Missing cost information is retained as a limitation rather than treated as zero.

### Report pages

1. **Executive Overview | Commercial Performance:** Financial KPIs, monthly revenue and estimated-profit trends, year-to-date revenue, and gross-profit comparisons across categories.
2. **Sales Detail | Product Analysis:** Units sold by product, detailed sales records, and category/date filters; conditional formatting helps identify negative and positive gross profit.
3. **Product Detail (drill-through):** Product-specific quantities, revenue, known costs, gross profit, and associated order records, accessible from the sales-detail report.

The original report was developed in Power BI. Screenshots and the `.pbix` file can be included separately in the portfolio repository, subject to data-sharing permissions.


## Findings, Recommendations, and Conclusions

### 6.1. Interactive Dashboards

Three Power BI report pages were developed to explore commercial performance from **January through June 2025**:

- **Executive Overview | Commercial Performance:** Core financial KPIs, monthly revenue and estimated-profit trends, year-to-date (YTD) revenue, and revenue and gross-profit comparisons by category.
- **Sales Detail | Product Analysis:** Product-level units sold and detailed sales records, with interactive category and date filters.
- **Product Detail:** A drill-through view for the selected product, including quantities, revenue, costs, and gross profit, with navigation from the sales report.

### 6.2. Key Findings

**Overall commercial performance**

The period generated **9.65 million USD in revenue**, **3.83 million USD in known product costs**, and **2.87 million USD in marketing expenditure**.

Calculated gross profit was **5.81 million USD**, corresponding to a **60.27% gross margin**. After deducting global marketing expenditure, estimated profit was **2.94 million USD**, equivalent to a **30.50% estimated margin**.

**Profitability by category**

The **Home (`Hogar`)** category recorded the highest gross margin (**62.52%**) and gross profit (**2.02 million USD**). **Fashion (`Moda`)** and **Electronics (`Electrónica`)** recorded gross margins of **59.49%** and **58.63%**, respectively.

**Profitability by product**

`Vacuum-Pro-Black` and `Sneakers-Urban-42` together accounted for approximately **51.4% of total calculated gross profit**. `Phone-Pro-128GB` had the highest gross margin among the identified products, at **96.04%**.

By contrast, `Laptop-Gaming-16GB` recorded a cumulative **gross loss of 91,419.48 USD** and a **-8.38% gross margin**. Its gross profit was negative in all six months, suggesting a persistent product-level profitability issue.

### 6.3. Business Recommendations

1. **Investigate `Laptop-Gaming-16GB` profitability:** Review realized selling prices, discounts, and unit costs to understand the recurring gross losses.
2. **Assess leading gross-profit contributors:** Monitor sales volume, margins, and demand stability for the products responsible for the largest share of gross profit.
3. **Strengthen financial monitoring:** Track category- and product-level gross margins regularly to identify unfavorable changes early.
4. **Improve data completeness:** Prioritize accurate quantity, unit-price, and cost records to increase confidence in reported profitability metrics.

### 6.4. Analytical Limitations

The dataset contains **50 orders with missing quantities** and **80 orders with insufficient information to calculate product costs**. These records were retained without unsupported imputation.

As a result, calculated gross profit **may be overstated**. Orders with unknown costs should **not** be interpreted as having a 100% gross margin.

Marketing expenditure was analyzed globally and **was not allocated across categories or products**. Product and category profitability are therefore assessed using gross profit and gross margin; estimated profit is reported only at the overall business level.

### 6.5. Overall Conclusion

This analysis evaluated commercial performance during the first half of 2025, identified differences in profitability across categories and products, and highlighted opportunities to review pricing, costs, and data quality.

Combining **Python** for data preparation and validation with **Power BI** for interactive reporting supported consistent KPI definitions and clearer exploration of business results.

The findings provide a foundation for prioritizing profitability reviews, strengthening financial monitoring, and informing commercial decisions, subject to the data limitations described above.


---

## Portfolio Deliverables

The GitHub repository is intended to contain this analysis notebook, the Power BI report (`.pbix`), selected dashboard screenshots, and documentation of the data preparation and validation workflow.

Before publishing, confirm that each dataset and report can be shared publicly, remove any credentials or private links, and document any database-access requirements. The original TripleTen submission link is intentionally excluded from this public-facing notebook.


**Dashboard access:** See the repository README and `powerbi/` directory for available report files and screenshots.
